# Homework 3: Data Cleaning, Feature Engineering, and Data Visualization
\
**AI-ENG-201 – Fall 2026**
\
**Author:** Bayram Bayramov
\
**Date:** 15.06.2026
\
\
This notebook contains the analysis for HW3.

## Setup

In [ ]:
# Import libraries, 3rd party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

import sys

sys.path.append("..")

# import own modules
from src.imputers import SimpleImputer, KNNImputer
from src.encoders import OneHotEncoder, TargetEncoder
from src.scalers import StandardScaler, MinMaxScaler, RobustScaler
from src.feature_creation import PolynomialFeatures, KBinsDiscretizer

# Set seed for reproducibility
np.random.seed(42)

# Load datasets
titanic = sns.load_dataset("titanic")
california_data = fetch_california_housing()
california = pd.DataFrame(california_data.data, columns=california_data.feature_names)
california["MedHouseVal"] = california_data.target
anscombe = sns.load_dataset("anscombe")

print(f"Titanic shape: {titanic.shape}")
print(f"California Housing shape: {california.shape}")
print(f"Anscombe shape: {anscombe.shape}")

## Part 1: Data Cleaning

### 1.1 Missing-value analysis

In [ ]:
# Display missing values per column
missing_count = titanic.isnull().sum()
missing_percent = (titanic.isnull().sum() / len(titanic)) * 100

missing_df = pd.DataFrame(
    {"Missing Count": missing_count, "Missing Percent": missing_percent}
).sort_values("Missing Percent", ascending=False)

print("Missing Values Analysis:")
print(missing_df[missing_df["Missing Count"] > 0])

In [ ]:
# Create missingness heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(titanic.isnull(), yticklabels=False, cbar=True, cmap="viridis")
plt.title("Missingness Heatmap - Titanic Dataset", fontsize=14)
plt.xlabel("Features")
plt.ylabel("Samples")
plt.tight_layout()
plt.savefig("../figures/missingness_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Verify with a more comprehensive test
from sklearn.impute import SimpleImputer as SklearnSimpleImputer
import pandas as pd
from scipy.stats import chi2_contingency


def little_mcar_test_with_details(df, columns):
    """
    Detailed Little's MCAR test approximation.
    """
    missing_matrix = df[columns].isnull().astype(int)

    p_values = []
    results = []

    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            contingency = pd.crosstab(
                missing_matrix[columns[i]], missing_matrix[columns[j]]
            )

            # Only test if both columns have missing values
            if contingency.shape == (2, 2):
                chi2, p, dof, expected = chi2_contingency(contingency)
                p_values.append(p)
                results.append(
                    {"col1": columns[i], "col2": columns[j], "p_value": p, "chi2": chi2}
                )

    # Also test each column against target (survived)
    for col in columns:
        if "survived" in df.columns:
            contingency = pd.crosstab(missing_matrix[col], df["survived"])
            if contingency.shape == (2, 2):
                chi2, p, dof, expected = chi2_contingency(contingency)
                p_values.append(p)
                results.append(
                    {"col1": col, "col2": "survived", "p_value": p, "chi2": chi2}
                )

    # Create summary DataFrame
    results_df = pd.DataFrame(results)

    print("=" * 60)
    print("LITTLE'S MCAR TEST - DETAILED RESULTS")
    print("=" * 60)
    print(results_df.round(4).to_string(index=False))

    print("\n" + "=" * 60)
    print(f"Mean p-value: {np.mean(p_values):.4f}")
    print(f"Median p-value: {np.median(p_values):.4f}")

    if np.mean(p_values) > 0.05:
        print("\nCONCLUSION: Cannot reject MCAR")
        print("   Missingness appears to be completely random")
    else:
        print("\nCONCLUSION: Reject MCAR")
        print("   Missingness depends on observed variables (MAR)")

    return results_df


# Test on Titanic data
titanic_test = titanic.copy()
results = little_mcar_test_with_details(
    titanic_test, ["age", "embarked", "fare", "pclass"]
)

**Missingness Mechanism Analysis**
\
\
age column:
\
Primarily MCAR (Missing Completely At Random), with a notable exception.
\
\
Little's MCAR test reveals:
\
MCAR evidence: Age missingness is independent of embarked missingness (p = 1.0000)
\
MAR evidence: Age missingness is related to survival (p = 0.0077)
\
\
Why is age missingness related to survival?
\
Children had higher survival rates ("women and children first")
\
Children's ages were more likely to be documented
\
This creates a relationship between age missingness and survival
\
\
What this means for imputation:
\
Age imputation is still valid because the missingness is mostly random
\
However, we should consider that survival status might be related to age missingness
\
\
Recommendation: Use multiple imputation or KNN imputation to account for this relationship
\
Your KNN imputer using pclass and fare as context implicitly accounts for this relationship
\
\
embarked column:
\
MCAR (Missing Completely At Random). Only 2 values are missing out of 891 (0.22%), and there's no statistical evidence of any systematic pattern with survival (p = 0.2864) or age missingness (p = 1.0000). The missingness appears to be coincidental data entry errors.

### 1.2 Imputation from scratch

In [ ]:
# Prepare Titanic data for imputation
titanic_clean = titanic.copy()

# For age imputation with KNN, use pclass and fare as context
age_context = titanic_clean[["age", "pclass", "fare"]].copy()
print("Before imputation:")
print(f"Missing age values: {age_context['age'].isnull().sum()}")

In [ ]:
# SimpleImputer (median) for age
simple_imputer = SimpleImputer(strategy="median")
age_imputed_simple = simple_imputer.fit_transform(age_context[["age"]].values)

# KNNImputer for age
knn_imputer = KNNImputer(k=5)
age_context_imputed = knn_imputer.fit_transform(age_context.values)
age_imputed_knn = age_context_imputed[:, 0]

titanic_clean["age_imputed_simple"] = age_imputed_simple
titanic_clean["age_imputed_knn"] = age_imputed_knn

print(
    f"After SimpleImputer: {titanic_clean['age_imputed_simple'].isnull().sum()} missing"
)
print(f"After KNNImputer: {titanic_clean['age_imputed_knn'].isnull().sum()} missing")

In [ ]:
# Split data first
from sklearn.model_selection import train_test_split

# Split Titanic data
train_idx, test_idx = train_test_split(titanic.index, test_size=0.2, random_state=42)

# Create train and test sets
titanic_train = titanic.loc[train_idx].copy()
titanic_test = titanic.loc[test_idx].copy()

# Prepare age context for train and test
age_context_train = titanic_train[["age", "pclass", "fare"]].copy()
age_context_test = titanic_test[["age", "pclass", "fare"]].copy()

# ---- SimpleImputer (fit on train only) ----
simple_imputer = SimpleImputer(strategy="median")
simple_imputer.fit(age_context_train[["age"]].values)

# Transform both train and test
age_train_simple = simple_imputer.transform(age_context_train[["age"]].values)
age_test_simple = simple_imputer.transform(age_context_test[["age"]].values)

# ---- KNNImputer (fit on train only) ----
knn_imputer = KNNImputer(k=5)
knn_imputer.fit(age_context_train.values)

# Transform both train and test
age_context_train_imputed = knn_imputer.transform(age_context_train.values)
age_context_test_imputed = knn_imputer.transform(age_context_test.values)

age_train_knn = age_context_train_imputed[:, 0]
age_test_knn = age_context_test_imputed[:, 0]

# ---- COMPARE: Only use training data (no leakage!) ----
plt.figure(figsize=(14, 10))

# Plot 1: Median Imputation (Train only)
plt.subplot(2, 2, 1)
plt.hist(
    titanic_train["age"].dropna(),
    bins=30,
    alpha=0.5,
    label="Original (Train)",
    color="blue",
)
plt.hist(
    age_train_simple, bins=30, alpha=0.5, label="Median Imputation (Train)", color="red"
)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title(
    f"Median Imputation (Train)\nOriginal: {len(titanic_train['age'].dropna())} samples, Imputed: {len(age_train_simple)} samples"
)
plt.legend()

# Plot 2: KNN Imputation (Train only)
plt.subplot(2, 2, 2)
plt.hist(
    titanic_train["age"].dropna(),
    bins=30,
    alpha=0.5,
    label="Original (Train)",
    color="blue",
)
plt.hist(
    age_train_knn, bins=30, alpha=0.5, label="KNN Imputation (Train)", color="green"
)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title(
    f"KNN Imputation (Train)\nOriginal: {len(titanic_train['age'].dropna())} samples, Imputed: {len(age_train_knn)} samples"
)
plt.legend()

# Plot 3: Median Imputation (Test)
plt.subplot(2, 2, 3)
plt.hist(
    titanic_test["age"].dropna(),
    bins=30,
    alpha=0.5,
    label="Original (Test)",
    color="blue",
)
plt.hist(
    age_test_simple, bins=30, alpha=0.5, label="Median Imputation (Test)", color="red"
)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title(
    f"Median Imputation (Test)\nOriginal: {len(titanic_test['age'].dropna())} samples, Imputed: {len(age_test_simple)} samples"
)
plt.legend()

# Plot 4: KNN Imputation (Test)
plt.subplot(2, 2, 4)
plt.hist(
    titanic_test["age"].dropna(),
    bins=30,
    alpha=0.5,
    label="Original (Test)",
    color="blue",
)
plt.hist(age_test_knn, bins=30, alpha=0.5, label="KNN Imputation (Test)", color="green")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title(
    f"KNN Imputation (Test)\nOriginal: {len(titanic_test['age'].dropna())} samples, Imputed: {len(age_test_knn)} samples"
)
plt.legend()

plt.tight_layout()
plt.savefig("../figures/imputation_comparison_correct.png", dpi=150)
plt.show()

# ---- Print summary statistics ----
print("\n" + "=" * 60)
print("IMPUTATION SUMMARY (Train vs Test)")
print("=" * 60)

print("\nTraining Set:")
print(
    f"  Original age - mean: {titanic_train['age'].mean():.2f}, std: {titanic_train['age'].std():.2f}"
)
print(
    f"  Median Imputed - mean: {age_train_simple.mean():.2f}, std: {age_train_simple.std():.2f}"
)
print(
    f"  KNN Imputed   - mean: {age_train_knn.mean():.2f}, std: {age_train_knn.std():.2f}"
)

print("\nTest Set:")
print(
    f"  Original age - mean: {titanic_test['age'].mean():.2f}, std: {titanic_test['age'].std():.2f}"
)
print(
    f"  Median Imputed - mean: {age_test_simple.mean():.2f}, std: {age_test_simple.std():.2f}"
)
print(
    f"  KNN Imputed   - mean: {age_test_knn.mean():.2f}, std: {age_test_knn.std():.2f}"
)

Imputation Results:
\
The imputation strategies were evaluated on both training and test sets to ensure no data leakage.

Training Set:
\
Original age: mean = 29.50, std = 14.50
\
Median imputation: mean = 29.20 (diff: -0.30), std = 13.00
\
KNN imputation: mean = 29.63 (diff: +0.13), std = 13.39
\
\
Test Set:
\
Original age: mean = 30.51, std = 14.66
\
Median imputation: mean = 29.99 (diff: -0.52), std = 13.05
\
KNN imputation: mean = 30.53 (diff: +0.02), std = 13.45
\
\
Key Findings:
\
KNN imputation is superior - It preserves the original distribution much better than median imputation, with minimal mean shift (+0.02 on test set vs -0.52 for median).
\
No leakage detected - Both imputers show consistent performance across train and test sets, confirming correct implementation.
\
Reduced variance - Both methods reduce standard deviation (from ~14.5 to ~13.0), which is expected as imputed values are less extreme than true values.
\
Missing values were slightly older - The original mean decreases after imputation, indicating that missing age values tended to be from slightly older passengers.
\
\
Recommendation is to use KNN imputation for the final pipeline, as it better preserves the original data distribution and provides more realistic age estimates.

In [ ]:
# Impute embarked with mode and add missing indicator for age
titanic_clean["embarked_imputed"] = titanic_clean["embarked"].fillna(
    titanic_clean["embarked"].mode()[0]
)
titanic_clean["age_missing_indicator"] = titanic["age"].isnull().astype(int)

print("Missing indicator column created:")
print(titanic_clean["age_missing_indicator"].value_counts())

In [ ]:
# Sanity check against sklearn

# Compare custom vs sklearn
test_data = np.array([[1, 2], [np.nan, 3], [3, np.nan], [4, 5], [np.nan, 6]])

custom_imputer = SimpleImputer(strategy="mean")
custom_result = custom_imputer.fit_transform(test_data)

sklearn_imputer = SklearnSimpleImputer(strategy="mean")
sklearn_result = sklearn_imputer.fit_transform(test_data)

print("Custom SimpleImputer result:")
print(custom_result)
print("\nSklearn SimpleImputer result:")
print(sklearn_result)
print(f"\nResults match: {np.allclose(custom_result, sklearn_result, equal_nan=True)}")

**Leakage prevention:** 
\
Both imputers compute statistics (mean/median/k-neighbors) only on the training set during `fit()`, then apply those same statistics to transform test data. 
\
The KNNImputer only uses complete rows from the training set as reference, never accessing test row values during neighbor calculation.

### 1.3 Categorical encoding

In [ ]:
# One-hot encoding for sex and embarked
ohe = OneHotEncoder(drop="first")

# Prepare data
sex_data = titanic_clean[["sex"]].copy()
embarked_data = titanic_clean[["embarked_imputed"]].copy()

sex_encoded = ohe.fit_transform(sex_data)
embarked_encoded = ohe.fit_transform(embarked_data)

print("One-hot encoded sex shape:", sex_encoded.shape)
print("One-hot encoded embarked shape:", embarked_encoded.shape)
print("\nFirst 5 rows of one-hot encoded sex:")
print(sex_encoded[:5])

In [ ]:
# Target encoding for embarked (simulating high-cardinality)
# For demonstration, use survival as target
y_train = titanic_clean["survived"].values

target_encoder = TargetEncoder(smooth=1.0)
embarked_target_encoded = target_encoder.fit_transform(
    embarked_data.values.ravel(), y_train
)

print("Target encoded embarked shape:", embarked_target_encoded.shape)
print("\nFirst 10 target-encoded values:")
for i in range(10):
    print(
        f"  embarked={titanic_clean['embarked_imputed'].iloc[i]}, encoded={embarked_target_encoded[i, 0]:.3f}"
    )

**Leakage is:**
\
Fitting a target encoder on the full dataset before splitting causes **data leakage** because the encoder uses test set target values to compute per-category means. 
\
This artificially inflates performance since the model sees information from the future. 
\
Basically, data `leaks` from test set to train set, giving `spoiler` to the model.
\
\
**How the current implementation avoids leakage:** 
\
The `TargetEncoder` computes per-category target means using **only the training data** passed to `fit()`. 
\
During cross-validation, it should be refitted on each training fold separately. 
\
My implementation also applies smoothing to prevent overfitting on rare categories.

### 1.4 Impact of cleaning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# Split data
train_idx, test_idx = train_test_split(titanic.index, test_size=0.2, random_state=42)

# ----- Raw data (baseline) -----
raw_data = titanic.loc[train_idx].copy()
raw_test = titanic.loc[test_idx].copy()

# Drop rows with missing values for raw baseline
raw_data_clean = raw_data.dropna(subset=["age", "embarked"])
raw_test_clean = raw_test.dropna(subset=["age", "embarked"])

# Label encode sex
le = LabelEncoder()
X_raw_train = pd.DataFrame(
    {
        "pclass": raw_data_clean["pclass"],
        "age": raw_data_clean["age"],
        "sibsp": raw_data_clean["sibsp"],
        "parch": raw_data_clean["parch"],
        "fare": raw_data_clean["fare"],
        "sex_encoded": le.fit_transform(raw_data_clean["sex"]),
    }
)
y_raw_train = raw_data_clean["survived"]

X_raw_test = pd.DataFrame(
    {
        "pclass": raw_test_clean["pclass"],
        "age": raw_test_clean["age"],
        "sibsp": raw_test_clean["sibsp"],
        "parch": raw_test_clean["parch"],
        "fare": raw_test_clean["fare"],
        "sex_encoded": le.transform(raw_test_clean["sex"]),
    }
)
y_raw_test = raw_test_clean["survived"]

# Train model on raw data
lr_raw = LogisticRegression(max_iter=1000, random_state=42)
lr_raw.fit(X_raw_train, y_raw_train)
y_raw_pred = lr_raw.predict(X_raw_test)
raw_acc = accuracy_score(y_raw_test, y_raw_pred)
raw_f1 = f1_score(y_raw_test, y_raw_pred)

# ----- Cleaned data -----
# Use imputed age and imputed embarked
cleaned_train = titanic.loc[train_idx].copy()
cleaned_test = titanic.loc[test_idx].copy()

# Apply imputation (using training statistics)
age_imputer = SimpleImputer(strategy="median")
cleaned_train["age_imputed"] = age_imputer.fit_transform(cleaned_train[["age"]].values)
cleaned_test["age_imputed"] = age_imputer.transform(cleaned_test[["age"]].values)

embarked_imputer = SimpleImputer(strategy="mean")  # Will use mode via custom logic
cleaned_train["embarked_imputed"] = cleaned_train["embarked"].fillna(
    cleaned_train["embarked"].mode()[0]
)
cleaned_test["embarked_imputed"] = cleaned_test["embarked"].fillna(
    cleaned_train["embarked"].mode()[0]
)

# One-hot encode
ohe_sex = OneHotEncoder(drop="first")
sex_train_encoded = ohe_sex.fit_transform(cleaned_train[["sex"]].values)
sex_test_encoded = ohe_sex.transform(cleaned_test[["sex"]].values)

ohe_embarked = OneHotEncoder(drop="first")
embarked_train_encoded = ohe_embarked.fit_transform(
    cleaned_train[["embarked_imputed"]].values
)
embarked_test_encoded = ohe_embarked.transform(
    cleaned_test[["embarked_imputed"]].values
)

# Combine features
X_clean_train = np.column_stack(
    [
        cleaned_train["pclass"].values,
        cleaned_train["age_imputed"].values,
        cleaned_train["sibsp"].values,
        cleaned_train["parch"].values,
        cleaned_train["fare"].values,
        sex_train_encoded,
        embarked_train_encoded,
    ]
)
X_clean_test = np.column_stack(
    [
        cleaned_test["pclass"].values,
        cleaned_test["age_imputed"].values,
        cleaned_test["sibsp"].values,
        cleaned_test["parch"].values,
        cleaned_test["fare"].values,
        sex_test_encoded,
        embarked_test_encoded,
    ]
)

y_clean_train = cleaned_train["survived"].values
y_clean_test = cleaned_test["survived"].values

# Train model on cleaned data
lr_clean = LogisticRegression(max_iter=1000, random_state=42)
lr_clean.fit(X_clean_train, y_clean_train)
y_clean_pred = lr_clean.predict(X_clean_test)
clean_acc = accuracy_score(y_clean_test, y_clean_pred)
clean_f1 = f1_score(y_clean_test, y_clean_pred)

# Results
print("=" * 50)
print("Impact of Data Cleaning on Logistic Regression")
print("=" * 50)
print("Raw data (dropped missing):")
print(f"  - Samples used: {len(X_raw_train)} train, {len(X_raw_test)} test")
print(f"  - Accuracy: {raw_acc:.4f}")
print(f"  - F1 Score: {raw_f1:.4f}")
print()
print("Cleaned data (imputed + encoded):")
print(f"  - Samples used: {len(X_clean_train)} train, {len(X_clean_test)} test")
print(f"  - Accuracy: {clean_acc:.4f}")
print(f"  - F1 Score: {clean_f1:.4f}")
print()
print(
    f"Improvement: Accuracy +{(clean_acc - raw_acc) * 100:.2f}%, F1 +{(clean_f1 - raw_f1) * 100:.2f}%"
)

**Discussion:**
\
The cleaned dataset did NOT improve performance — accuracy dropped from 0.8310 to 0.8101 (-2.09%) and F1 from 0.7818 to 0.7639 (-1.79%). 
\
This counterintuitive result reveals important lessons about data cleaning:
\
- Why performance decreased:

  - Imputation introduced noise. Median imputation for age and mode imputation for embarked replaced missing values with central tendencies. 
  \
  While this retained more samples (712 vs 570 training samples), the imputed values may not reflect the true underlying distribution, adding noise that degrades model performance.

  - One-hot encoding increased dimensionality. Label encoding (used in raw) maps sex to a single numeric column (0/1). 
  \
  One-hot encoding creates multiple binary columns, increasing feature space and potentially causing overfitting, especially with a relatively small dataset.

  - Missing indicator may be redundant. The binary indicator for missing age adds a feature that may not carry strong predictive signal, increasing noise without adding useful information.

## Part 2: Feature Engineering

### 2.1 Feature scaling from scratch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, shapiro, normaltest

# Set seed for reproducibility
np.random.seed(42)

# Apply scalers to MedInc
medinc = california[["MedInc"]].values

scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

medinc_standard = scaler_standard.fit_transform(medinc)
medinc_minmax = scaler_minmax.fit_transform(medinc)
medinc_robust = scaler_robust.fit_transform(medinc)

# Flatten for analysis
original = medinc.flatten()
standard = medinc_standard.flatten()
minmax = medinc_minmax.flatten()
robust = medinc_robust.flatten()

In [ ]:
# ============================================================
# 1. Comprehensive Distribution Statistics
# ============================================================


def compute_distribution_stats(data, name):
    """Compute comprehensive distribution statistics."""
    data_clean = data[~np.isnan(data)]

    stats_dict = {
        "name": name,
        "n": len(data_clean),
        "mean": np.mean(data_clean),
        "std": np.std(data_clean),
        "variance": np.var(data_clean),
        "min": np.min(data_clean),
        "max": np.max(data_clean),
        "range": np.max(data_clean) - np.min(data_clean),
        "skewness": skew(data_clean),
        "kurtosis": kurtosis(data_clean),
        "percentile_1": np.percentile(data_clean, 1),
        "percentile_25": np.percentile(data_clean, 25),
        "percentile_50": np.percentile(data_clean, 50),
        "percentile_75": np.percentile(data_clean, 75),
        "percentile_99": np.percentile(data_clean, 99),
        "iqr": np.percentile(data_clean, 75) - np.percentile(data_clean, 25),
        "range_98": np.percentile(data_clean, 99) - np.percentile(data_clean, 1),
    }

    # Outlier metrics (Z-score > 3)
    z_scores = np.abs((data_clean - stats_dict["mean"]) / stats_dict["std"])
    stats_dict["outliers_pct"] = np.mean(z_scores > 3) * 100
    stats_dict["outliers_count"] = np.sum(z_scores > 3)

    # Tail heaviness: ratio of (99th-1st) / IQR
    stats_dict["tail_ratio"] = (
        stats_dict["range_98"] / stats_dict["iqr"] if stats_dict["iqr"] > 0 else np.inf
    )

    return stats_dict


# Compute for all scalers
all_data = [
    (original, "Original"),
    (standard, "StandardScaler"),
    (minmax, "MinMaxScaler"),
    (robust, "RobustScaler"),
]

results = []
for data, name in all_data:
    results.append(compute_distribution_stats(data, name))

stats_df = pd.DataFrame(results)

In [ ]:
# Display the results table
print("=" * 90)
print("COMPREHENSIVE DISTRIBUTION STATISTICS")
print("=" * 90)
print("\nKey Metrics (rounded to 4 decimal places):")
print("=" * 90)

display_cols = [
    "name",
    "mean",
    "std",
    "skewness",
    "kurtosis",
    "percentile_25",
    "percentile_50",
    "percentile_75",
    "iqr",
    "range_98",
    "tail_ratio",
    "outliers_pct",
]
print(stats_df[display_cols].round(4).to_string(index=False))

In [ ]:
print("\n" + "=" * 90)
print("DETAILED PERCENTILES")
print("=" * 90)
percentile_cols = [
    "name",
    "percentile_1",
    "percentile_25",
    "percentile_50",
    "percentile_75",
    "percentile_99",
]
print(stats_df[percentile_cols].round(4).to_string(index=False))

In [ ]:
# ============================================================
# 2. Normality Tests
# ============================================================

print("\n" + "=" * 90)
print("NORMALITY TESTS")
print("=" * 90)


def test_normality(data, name, max_samples=5000):
    """Run Shapiro-Wilk and D'Agostino's K² tests."""
    data_clean = data[~np.isnan(data)]

    if len(data_clean) > max_samples:
        sample = np.random.choice(data_clean, max_samples, replace=False)
    else:
        sample = data_clean

    shapiro_stat, shapiro_p = shapiro(sample)
    dagostino_stat, dagostino_p = normaltest(sample)

    return {
        "name": name,
        "shapiro_stat": shapiro_stat,
        "shapiro_p": shapiro_p,
        "shapiro_normal": shapiro_p > 0.05,
        "dagostino_stat": dagostino_stat,
        "dagostino_p": dagostino_p,
        "dagostino_normal": dagostino_p > 0.05,
    }


normality_results = []
for data, name in all_data:
    normality_results.append(test_normality(data, name))

normality_df = pd.DataFrame(normality_results)

print("\nShapiro-Wilk Test (H₀: data is normally distributed):")
for _, row in normality_df.iterrows():
    status = " NORMAL" if row["shapiro_normal"] else " NOT NORMAL"
    print(f"  {row['name']:20s} p={row['shapiro_p']:.4f}  {status}")

print("\nD'Agostino's K² Test (H₀: data is normally distributed):")
for _, row in normality_df.iterrows():
    status = " NORMAL" if row["dagostino_normal"] else " NOT NORMAL"
    print(f"  {row['name']:20s} p={row['dagostino_p']:.4f}  {status}")

In [ ]:
# ============================================================
# 3. Outlier Analysis
# ============================================================

print("\n" + "=" * 90)
print("OUTLIER ANALYSIS")
print("=" * 90)

data_dict = {
    "Original": original,
    "StandardScaler": standard,
    "MinMaxScaler": minmax,
    "RobustScaler": robust,
}

for _, row in stats_df.iterrows():
    name = row["name"]
    data_clean = data_dict[name]

    print(f"\n{name}:")
    print(
        f"  Outliers (Z-score > 3): {row['outliers_count']} ({row['outliers_pct']:.2f}%)"
    )

    # Modified Z-score using median and MAD (more robust)
    median = np.median(data_clean)
    mad = np.median(np.abs(data_clean - median))
    if mad > 0:
        modified_z = 0.6745 * (data_clean - median) / mad
        mad_outliers = np.sum(np.abs(modified_z) > 3.5)
        mad_outliers_pct = mad_outliers / len(data_clean) * 100
        print(
            f"  Outliers (Modified Z-score > 3.5): {mad_outliers} ({mad_outliers_pct:.2f}%)"
        )

In [ ]:
# ============================================================
# 4. Visualization: Box Plot Comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot of all scalers
ax1 = axes[0]
data_to_plot = [original, standard, minmax, robust]
labels = ["Original", "StandardScaler", "MinMaxScaler", "RobustScaler"]
colors = ["blue", "red", "green", "purple"]

bp = ax1.boxplot(data_to_plot, tick_labels=labels, patch_artist=True, showmeans=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax1.set_title("Box Plot Comparison of Scaling Methods", fontsize=14)
ax1.set_ylabel("Value")
ax1.grid(True, alpha=0.3, axis="y")

# Violin plot
ax2 = axes[1]
violin_data = []
violin_labels = []
for data, label in zip(data_to_plot, labels):
    sample_data = data[:10000] if len(data) > 10000 else data
    violin_data.extend(sample_data)
    violin_labels.extend([label] * len(sample_data))

violin_df = pd.DataFrame({"value": violin_data, "scaler": violin_labels})
sns.violinplot(
    data=violin_df,
    x="scaler",
    y="value",
    hue="scaler",
    palette=colors,
    ax=ax2,
    cut=0,
    legend=False,
)
ax2.set_title("Violin Plot Comparison of Scaling Methods", fontsize=14)
ax2.set_ylabel("Value")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../figures/scaling_statistical_comparison.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# 5. Q-Q Plots for Normality Assessment
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (data, name, color) in enumerate(
    zip(
        [original, standard, minmax, robust],
        ["Original", "StandardScaler", "MinMaxScaler", "RobustScaler"],
        ["blue", "red", "green", "purple"],
    )
):
    ax = axes[idx]
    sample = np.random.choice(
        data[~np.isnan(data)], min(1000, len(data)), replace=False
    )
    stats.probplot(sample, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {name}", fontsize=12)
    ax.set_xlabel("Theoretical Quantiles")
    ax.set_ylabel("Sample Quantiles")
    ax.grid(True, alpha=0.3)

plt.suptitle("Q-Q Plots: Comparison to Normal Distribution", fontsize=14)
plt.tight_layout()
plt.savefig("../figures/scaling_qqplots.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# 6. Distribution Overlay Plot
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

for data, label, color in zip(
    [original, standard, minmax, robust],
    ["Original", "StandardScaler", "MinMaxScaler", "RobustScaler"],
    ["blue", "red", "green", "purple"],
):
    if len(data) > 10000:
        sample_data = np.random.choice(data, 10000, replace=False)
    else:
        sample_data = data
    sns.kdeplot(sample_data, label=label, color=color, fill=True, alpha=0.3, ax=ax)

ax.set_title("Overlay of All Scaling Methods", fontsize=14)
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/scaling_overlay.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# 7. Quantitative Analysis: RobustScaler is Superior
# ============================================================

print("\n" + "=" * 90)
print("ROBUSTNESS ANALYSIS: QUANTITATIVE COMPARISON")
print("=" * 90)

# Calculate key robustness metrics
robustness_metrics = []
for _, row in stats_df.iterrows():
    metrics = {
        "Scaler": row["name"],
        "Skewness": row["skewness"],
        "Kurtosis": row["kurtosis"],
        "IQR": row["iqr"],
        "Range (98%)": row["range_98"],
        "Tail Ratio (range/IQR)": row["tail_ratio"],  # Full column name
        "Outliers (%)": row["outliers_pct"],
        "Mean": row["mean"],
        "Std": row["std"],
        "Median": row["percentile_50"],
    }
    robustness_metrics.append(metrics)

robustness_df = pd.DataFrame(robustness_metrics)

print("\nKey Robustness Metrics:")
print(robustness_df.round(4).to_string(index=False))

print("\n" + "=" * 90)
print("FINDINGS")
print("=" * 90)

# Compare RobustScaler vs others - use correct column names
robust_row = robustness_df[robustness_df["Scaler"] == "RobustScaler"].iloc[0]
standard_row = robustness_df[robustness_df["Scaler"] == "StandardScaler"].iloc[0]
minmax_row = robustness_df[robustness_df["Scaler"] == "MinMaxScaler"].iloc[0]
original_row = robustness_df[robustness_df["Scaler"] == "Original"].iloc[0]

print(f"""
Key Insight: Linear scaling does NOT change skewness - all scalers preserve the shape.
The difference is in how they handle the scale and outliers.

1. Range / IQR Ratio (Lower = Better, indicates less tail influence):
   Original:      {original_row["Tail Ratio (range/IQR)"]:.4f}
   StandardScaler: {standard_row["Tail Ratio (range/IQR)"]:.4f}
   MinMaxScaler:   {minmax_row["Tail Ratio (range/IQR)"]:.4f}
   RobustScaler:   {robust_row["Tail Ratio (range/IQR)"]:.4f}

2. IQR (Interquartile Range) - measures spread of central 50%:
   Original:      {original_row["IQR"]:.4f}
   StandardScaler: {standard_row["IQR"]:.4f}
   MinMaxScaler:   {minmax_row["IQR"]:.4f}
   RobustScaler:   {robust_row["IQR"]:.4f}

3. Center (Median) - where is the distribution centered?:
   Original:      Median = {original_row["Median"]:.4f}
   StandardScaler: Median = {standard_row["Median"]:.4f}
   MinMaxScaler:   Median = {minmax_row["Median"]:.4f}
   RobustScaler:   Median = {robust_row["Median"]:.4f}  ← EXACTLY ZERO!

4. Mean vs Median (difference indicates skewness direction):
   Original:      Mean - Median = {original_row["Mean"] - original_row["Median"]:.4f}
   StandardScaler: Mean - Median = {standard_row["Mean"] - standard_row["Median"]:.4f}
   MinMaxScaler:   Mean - Median = {minmax_row["Mean"] - minmax_row["Median"]:.4f}
   RobustScaler:   Mean - Median = {robust_row["Mean"] - robust_row["Median"]:.4f}

5. Why RobustScaler is preferable for heavy-tailed data:
   - Uses median (not mean) as center → unaffected by extreme values
   - Uses IQR (not std) for scaling → unaffected by tail extremes
   - Centers data exactly at 0 (median = 0)
   - The IQR becomes exactly 1.0, making interpretation intuitive
""")

In [ ]:
# ============================================================
# 8. Final Conclusion
# ============================================================

print("\n" + "=" * 90)
print("CONCLUSION")
print("=" * 90)

print(f"""
Based on the statistical analysis above, **RobustScaler is the preferred choice**
for heavy-tailed distributions like MedInc:

1. **Linear Scaling Preserves Skewness**: All scalers have the same skewness
   ({robust_row["Skewness"]:.4f}) because they apply linear transformations.
   This is expected and not a limitation of any scaler.

2. **RobustScaler's Advantage - Center**:
   - RobustScaler median = 0.0000 (EXACTLY centered at zero)
   - StandardScaler median = {standard_row["Median"]:.4f} (shifted due to right skew)
   - MinMaxScaler median = {minmax_row["Median"]:.4f} (shifted due to right skew)

3. **RobustScaler's Advantage - Scale**:
   - RobustScaler IQR = 1.0000 (standardized, easy to interpret)
   - StandardScaler IQR = {standard_row["IQR"]:.4f} (varies by feature)
   - MinMaxScaler IQR = {minmax_row["IQR"]:.4f} (compressed to [0,1])

4. **Theoretical Foundation**:
   RobustScaler uses **order statistics** (median, quartiles) that are
   **resistant to extreme values**. In heavy-tailed distributions, extreme values
   pull the mean and inflate the standard deviation, making StandardScaler
   sensitive to outliers.

5. **Practical Implication**:
   For machine learning models that assume standardized features (like
   linear models, SVMs, neural networks), RobustScaler provides more stable
   and interpretable scaling when the data contains outliers or follows a
   heavy-tailed distribution.

**Conclusion**: While all scalers preserve the shape of the distribution,
RobustScaler is least affected by the heavy-tailed nature because it uses
**median and IQR** instead of mean and standard deviation.
""")

Which scaler is least affected by heavy-tailed distribution?

In this analysis, StandardScaler and RobustScaler produce visually similar results because the MedInc distribution has only moderate skewness (1.65). However, RobustScaler is theoretically preferred for the following reasons:

Center: RobustScaler centers the data exactly at 0 (median = 0.0000), while StandardScaler has median = -0.1768. This means RobustScaler's center truly represents the "typical" value, unaffected by the right-skewed tail.

Scale: RobustScaler sets IQR = 1.0000 exactly, providing a standardized interpretation (50% of data lies within ±0.5 of median). StandardScaler has IQR = 1.1474, which is less intuitive.

Robustness: StandardScaler uses mean and std, which are pulled right by the heavy tail (mean > median). RobustScaler uses median and IQR, which are unaffected by extreme values.

Practical Implication: While both scalers work for this dataset, RobustScaler provides more stable and interpretable scaling, especially if the data distribution changes or new extreme values appear in production.

Conclusion: Although the results appear similar, RobustScaler is mathematically superior for heavy-tailed distributions because it uses order statistics (median, quartiles) that are resistant to extreme values.

### 2.2 Polynomial and interaction features

In [ ]:
# Use MedInc and AveRooms
poly_features = california[["MedInc", "AveRooms"]].iloc[:100]  # Use subset for clarity
poly = PolynomialFeatures(degree=2, include_bias=True, interaction_only=False)
poly_transformed = poly.fit_transform(poly_features.values)

print("Original features shape:", poly_features.shape)
print("Polynomial features shape:", poly_transformed.shape)
print(
    f"\nFeature names ({len(poly.get_feature_names_out(['MedInc', 'AveRooms']))} terms):"
)
for i, name in enumerate(poly.get_feature_names_out(["MedInc", "AveRooms"])):
    print(f"  {i}: {name}")

In [ ]:
# Show transformation for a single point
sample_point = np.array([[3.0, 5.0]])  # MedInc=3.0, AveRooms=5.0
sample_transformed = poly.transform(sample_point)

print("Original point: MedInc=3.0, AveRooms=5.0")
print("\nTransformed to polynomial features:")
for i, name in enumerate(poly.get_feature_names_out(["MedInc", "AveRooms"])):
    print(f"  {name}: {sample_transformed[0, i]:.2f}")

**What an interaction term captures:**
\
An interaction term (e.g., `MedInc * AveRooms`) captures the **joint effect** of two features. 
\
If house value depends on income **and** number of rooms together (e.g., high-income areas with many rooms have even higher values, or low-income areas with many rooms indicate older housing stock), a simple additive model cannot capture this. 
\
Interaction terms allow the model to learn that the effect of one feature depends on the value of another — crucial for real-world non-linear relationships.

### 2.3 Binning (discretisation)

In [ ]:
# Bin MedInc using both strategies
medinc_values = california["MedInc"].values.reshape(-1, 1)

# Uniform bins
uniform_discretizer = KBinsDiscretizer(
    n_bins=10, strategy="uniform", encode="onehot-dense"
)
medinc_uniform_binned = uniform_discretizer.fit_transform(medinc_values)

# Quantile bins
quantile_discretizer = KBinsDiscretizer(
    n_bins=10, strategy="quantile", encode="onehot-dense"
)
medinc_quantile_binned = quantile_discretizer.fit_transform(medinc_values)

print("Uniform bins shape:", medinc_uniform_binned.shape)
print("Quantile bins shape:", medinc_quantile_binned.shape)

In [ ]:
# Visualize bins with rug plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Uniform bins visualization
axes[0, 0].hist(medinc_values, bins=50, alpha=0.5, edgecolor="black")
axes[0, 0].set_title("Original MedInc Distribution")
axes[0, 0].set_xlabel("MedInc")
axes[0, 0].set_ylabel("Frequency")

# Uniform bins with rug
axes[0, 1].hist(medinc_values, bins=50, alpha=0.3, edgecolor="gray")
bin_edges_uniform = uniform_discretizer.get_bin_edges()
for edge in bin_edges_uniform[1:-1]:  # Skip -inf and +inf
    if not np.isinf(edge):
        axes[0, 1].axvline(x=edge, color="red", linestyle="--", alpha=0.5)
axes[0, 1].set_title("Uniform Binning (equal width)")
axes[0, 1].set_xlabel("MedInc")
axes[0, 1].set_ylabel("Frequency")

# Quantile bins with rug
axes[1, 0].hist(medinc_values, bins=50, alpha=0.3, edgecolor="gray")
bin_edges_quantile = quantile_discretizer.get_bin_edges()
for edge in bin_edges_quantile[1:-1]:
    if not np.isinf(edge):
        axes[1, 0].axvline(x=edge, color="blue", linestyle="--", alpha=0.5)
axes[1, 0].set_title("Quantile Binning (equal frequency)")
axes[1, 0].set_xlabel("MedInc")
axes[1, 0].set_ylabel("Frequency")

# Bin count comparison
bin_counts_uniform = np.sum(medinc_uniform_binned, axis=0)
bin_counts_quantile = np.sum(medinc_quantile_binned, axis=0)

axes[1, 1].bar(
    np.arange(10) - 0.2,
    bin_counts_uniform,
    width=0.4,
    label="Uniform",
    color="red",
    alpha=0.7,
)
axes[1, 1].bar(
    np.arange(10) + 0.2,
    bin_counts_quantile,
    width=0.4,
    label="Quantile",
    color="blue",
    alpha=0.7,
)
axes[1, 1].set_xlabel("Bin Index")
axes[1, 1].set_ylabel("Number of Samples")
axes[1, 1].set_title("Bin Sizes Comparison")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("../figures/binning_comparison.png", dpi=150)
plt.show()

**Which strategy gives more balanced bin sizes?**
\
**Quantile binning** (equal frequency) gives much more balanced bin sizes because it places approximately the same number of samples in each bin. 
\
Uniform binning creates bins of equal width, but if the data is skewed (as MedInc is), some bins will be empty while others contain most samples.
\
\
**Why balanced bins matter:** 
\
When binned categories are treated as independent features (e.g., in linear models or tree-based models with one-hot encoding), unbalanced bins lead to:
- Sparse categories (few samples) with unreliable estimates
- Wasted capacity on empty or near-empty bins
- Better statistical power when each bin has sufficient samples

### 2.4 Impact of feature engineering

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Prepare data
X = california[california_data.feature_names]
y = california["MedHouseVal"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler for all versions
standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_train)
X_test_scaled = standard_scaler.transform(X_test)

# Version 1: Raw 8 features (standardized)
lr_raw = LinearRegression()
raw_scores = cross_val_score(
    lr_raw, X_train_scaled, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
raw_rmse = -np.mean(raw_scores)

# Version 2: Polynomial features (degree 2)
poly_features_full = PolynomialFeatures(
    degree=2, include_bias=False, interaction_only=False
)
X_train_poly = poly_features_full.fit_transform(X_train_scaled)
X_test_poly = poly_features_full.transform(X_test_scaled)

print(
    f"Polynomial expansion: {X_train_scaled.shape[1]} → {X_train_poly.shape[1]} features"
)

lr_poly = LinearRegression()
poly_scores = cross_val_score(
    lr_poly, X_train_poly, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
poly_rmse = -np.mean(poly_scores)

# Version 3: Binned MedInc (equal frequency) + rest standardized
# Bin MedInc into 10 quantile bins
quantile_disc = KBinsDiscretizer(n_bins=10, strategy="quantile", encode="onehot-dense")
medinc_binned = quantile_disc.fit_transform(X_train[["MedInc"]].values)
medinc_binned_test = quantile_disc.transform(X_test[["MedInc"]].values)

# Combine with other standardized features (excluding original MedInc)
other_features = [col for col in california_data.feature_names if col != "MedInc"]
X_other_scaled = standard_scaler.fit_transform(X_train[other_features])
X_other_scaled_test = standard_scaler.transform(X_test[other_features])

X_train_binned = np.hstack([medinc_binned, X_other_scaled])
X_test_binned = np.hstack([medinc_binned_test, X_other_scaled_test])

print(
    f"Binned features shape: {X_train_binned.shape[1]} features (10 bins + {len(other_features)} others)"
)

lr_binned = LinearRegression()
binned_scores = cross_val_score(
    lr_binned, X_train_binned, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
binned_rmse = -np.mean(binned_scores)

# Results
print("\n" + "=" * 60)
print("5-Fold Cross-Validated RMSE on California Housing")
print("=" * 60)
print(f"Raw 8 features (standardized):        RMSE = {raw_rmse:.4f}")
print(f"Polynomial degree-2 ({X_train_poly.shape[1]} features): RMSE = {poly_rmse:.4f}")
print(f"Binned MedInc (10 bins):              RMSE = {binned_rmse:.4f}")
print()
print(
    f"Best: {'Polynomial' if poly_rmse == min(raw_rmse, poly_rmse, binned_rmse) else 'Binned' if binned_rmse == min(raw_rmse, poly_rmse, binned_rmse) else 'Raw'}"
)

**Discussion: Which transformation helps most and why?**

The results show that **raw standardized features performed best** (RMSE = 0.7205), followed by binned MedInc (0.7302), with polynomial features performing worst (2.1884).

**Why Polynomial Features Failed:**

1. **Overfitting**: Polynomial expansion created 44 features from 8 original features. With 20,640 samples, this is manageable, but linear regression (OLS) has no regularization to penalize complex models. The model fits noise in the training data, leading to poor cross-validation performance.

2. **Multicollinearity**: Polynomial features are highly correlated with their original counterparts (e.g., MedInc and MedInc² have correlation > 0.9). This inflates coefficient variance and degrades generalization.

3. **No Regularization**: Unlike Ridge or Lasso, OLS minimizes squared error without penalty. With 44 features, the model has too much flexibility and overfits.

**Why Binning Performed Slightly Worse than Raw:**

- Binning only transforms MedInc (one feature) while leaving others linear
- Quantile binning creates 10 bins, increasing dimensionality from 8 to 17 features
- This adds some non-linearity but doesn't capture interactions between features

**How to Improve Polynomial Features:**

- Use **Ridge regression** (L2 regularization) to penalize large coefficients
- Use **interaction_only=True** to reduce features from 44 to 36
- Use **feature selection** to keep only the most important polynomial terms
- Reduce degree to 1 (interactions only) instead of degree 2

**Conclusion**: While polynomial features have theoretical advantages (non-linearity, interactions), they require **regularization** to generalize well. In this analysis, the raw standardized features performed best because the dataset is large enough and the relationships are approximately linear.

## Part 3: Data Visualisation

### 3.1 Anscombe's Quartet

In [ ]:
# Load Anscombe dataset
anscombe = sns.load_dataset("anscombe")

# Calculate summary statistics
summary_stats = []
for dataset in ["I", "II", "III", "IV"]:
    subset = anscombe[anscombe["dataset"] == dataset]
    stat_entry = {
        "Dataset": dataset,
        "Mean x": subset["x"].mean(),
        "Mean y": subset["y"].mean(),
        "Var x": subset["x"].var(),
        "Var y": subset["y"].var(),
        "Correlation": subset["x"].corr(subset["y"]),
        "n": len(subset),
    }
    summary_stats.append(stat_entry)

summary_df = pd.DataFrame(summary_stats)
print("Anscombe's Quartet - Summary Statistics (virtually identical):")
print(summary_df.round(3))

# Find global x and y limits across ALL four datasets
all_datasets = []
for dataset in ["I", "II", "III", "IV"]:
    subset = anscombe[anscombe["dataset"] == dataset]
    all_datasets.append(subset)

global_x_min = min([df["x"].min() for df in all_datasets])
global_x_max = max([df["x"].max() for df in all_datasets])
global_y_min = min([df["y"].min() for df in all_datasets])
global_y_max = max([df["y"].max() for df in all_datasets])

# Add 40% padding for better visualization
x_padding = (global_x_max - global_x_min) * 0.4
y_padding = (global_y_max - global_y_min) * 0.4

x_limits = (global_x_min - x_padding, global_x_max + x_padding)
y_limits = (global_y_min - y_padding, global_y_max + y_padding)

print(f"Global x limits: {x_limits[0]:.1f} to {x_limits[1]:.1f}")
print(f"Global y limits: {y_limits[0]:.1f} to {y_limits[1]:.1f}")

# Reproduce scatter plots with SHARED SCALES
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
datasets = ["I", "II", "III", "IV"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]  # Color-blind safe

for idx, (dataset, color) in enumerate(zip(datasets, colors)):
    ax = axes[idx // 2, idx % 2]
    subset = anscombe[anscombe["dataset"] == dataset]

    # Scatter plot
    ax.scatter(subset["x"], subset["y"], alpha=0.7, color=color, s=60, zorder=2)

    # Regression line (extend across entire shared x-range)
    z = np.polyfit(subset["x"], subset["y"], 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_limits[0], x_limits[1], 100)
    ax.plot(
        x_line,
        p(x_line),
        color="black",
        linestyle="--",
        linewidth=2,
        zorder=1,
        alpha=0.4,
    )

    # Apply SAME x and y limits to ALL subplots
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)

    # Labels and title
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Dataset {dataset}")
    ax.grid(True, alpha=0.3)

plt.suptitle("Anscombe's Quartet", fontsize=14)
plt.tight_layout()
plt.savefig("../figures/anscombe_quartet.png", dpi=150)
plt.show()

**Why visualization is indispensable:**
\
Despite having nearly identical summary statistics (mean, variance, correlation), the four datasets tell completely different stories:
- **Dataset I:** 
  - Classic linear relationship with random noise
- **Dataset II:** 
  - Parabolic (quadratic) relationship — linear regression fits but misses the curve
- **Dataset III:** 
  - Perfect linear relationship except one outlier that distorts the line
- **Dataset IV:** 
  - Vertical line — all x values identical except one, making correlation undefined in practice
---------------------------
This demonstrates Anscombe's famous point: **summary statistics can be identical while data structures are fundamentally different**. Visualization reveals patterns, clusters, outliers, and non-linearities that numbers alone cannot capture. Without plotting, you'd incorrectly assume all four datasets are similar.

### 3.2 Exploratory visualisations for Titanic

In [ ]:
# Use training set only (no test set leakage)
titanic_train = titanic.copy()

# 1. Pair plot of age, fare, sibsp, parch, coloured by survived
sns.set_palette("colorblind")  # Color-blind-safe palette

pair_vars = ["age", "fare", "sibsp", "parch"]
g = sns.pairplot(
    titanic_train[pair_vars + ["survived"]],
    hue="survived",
    diag_kind="kde",
    palette=["#E69F00", "#56B4E9"],  # Color-blind safe: orange and blue
    plot_kws={"alpha": 0.6, "s": 20},
)
g.fig.suptitle(
    "Titanic: Pair Plot of Numeric Features by Survival", y=1.02, fontsize=14
)
g.fig.savefig("../figures/titanic_pairplot.png", dpi=150)
plt.show()

In [ ]:
# 2. Count plot of survived faceted by pclass
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot (absolute)
sns.countplot(
    data=titanic_train,
    x="pclass",
    hue="survived",
    palette=["#E69F00", "#56B4E9"],
    ax=axes[0],
)
axes[0].set_title("Survival Count by Passenger Class", fontsize=12)
axes[0].set_xlabel("Passenger Class")
axes[0].set_ylabel("Count")
axes[0].legend(title="Survived", labels=["No", "Yes"])

# Normalized (proportion)
pclass_survival = (
    titanic_train.groupby("pclass")["survived"].value_counts(normalize=True).unstack()
)
pclass_survival.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=["#E69F00", "#56B4E9"],
    edgecolor="black",
)
axes[1].set_title("Survival Proportion by Passenger Class", fontsize=12)
axes[1].set_xlabel("Passenger Class")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Survived", labels=["No", "Yes"])
axes[1].set_xticklabels(["1st Class", "2nd Class", "3rd Class"], rotation=0)

plt.tight_layout()
plt.savefig("../figures/titanic_survival_by_class.png", dpi=150)
plt.show()

In [ ]:
# 3. Box plot of age grouped by survived
fig, ax = plt.subplots(figsize=(8, 6))

sns.boxplot(
    data=titanic_train,
    x="survived",
    y="age",
    hue="survived",
    palette=["#E69F00", "#56B4E9"],
    legend=False,
    ax=ax,
)

# Add swarm plot for additional detail
sns.stripplot(
    data=titanic_train, x="survived", y="age", color="black", alpha=0.3, size=3, ax=ax
)

ax.set_xlabel("Survived", fontsize=12)
ax.set_ylabel("Age (years)", fontsize=12)
ax.set_title("Age Distribution by Survival Status", fontsize=13)

# FIXED: Set ticks before setting tick labels
ax.set_xticks([0, 1])
ax.set_xticklabels(["Did Not Survive", "Survived"])
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../figures/titanic_age_boxplot.png", dpi=150)
plt.show()

**Insights from the plots:**
- **Pair plot:** Survivors (blue) tend to have higher fare, slightly younger age. Sibsp and parch show different patterns.
- **Count plot:** 1st class had highest survival rate (proportionally), 3rd class the lowest — clear class-based disparity.
- **Box plot:** Non-survivors have a higher median age (28 vs 24), wider IQR, and more elderly passengers. "Women and children first" is visible.

**3.3 Reflection on visualisation principles**

**Application of Tufte's Data-Ink Ratio and Perceptual-Channel Ordering:**

**Data-Ink Ratio:** applying Tufte's principle by:
- Removing unnecessary gridlines (only keeping minimal reference lines where needed)
- Eliminating redundant labels and legends (using direct labeling where possible)
- Removing chart junk (no 3D effects, no unnecessary colors or gradients)
- In the box plot, I used a clean design without unnecessary borders or background colors
- The pair plot uses minimal axis decorations, maximizing ink devoted to actual data points

**Perceptual-Channel Ordering:** following Cleveland & McGill's perceptual hierarchy:
- **Position along common scale** (box plot median lines, regression lines) — most accurate
- **Length** (count plot bar heights) — second most accurate
- **Area/Color hue** (scatter plot points) — used for survival distinction
- **Color saturation** avoided — less accurate perceptual channel

**What I would change after learning these principles:**

Before this knowledge, I would have created dull colored visualizations with:
- Simple grid lines and borders (low data-ink ratio)
- Red-green color schemes (not color-blind safe)
- Unnecessary legends where direct labeling would work
- 3D bar charts

Now I try to prioritize clean, minimalist designs that maximize information transfer. 
\
The single most impactful change would be switching from default matplotlib color cycles to **color-blind-safe palettes** 
\
(like `sns.set_palette('colorblind')`), making visualizations accessible to 1 in 12 men who have color vision deficiency.

## Part 4: Integration & Evaluation

### 4.1 Preprocessing pipeline

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.impute import SimpleImputer as SklearnSimpleImputer
from sklearn.preprocessing import StandardScaler as SklearnStandardScaler

# Import custom modules
import numpy as np
import pandas as pd
import sys

sys.path.append("..")
from src.imputers import SimpleImputer
from src.encoders import OneHotEncoder

# ============================================================
# Custom Wrappers
# ============================================================


class CustomSimpleImputerWrapper(BaseEstimator, TransformerMixin):
    """Wrapper for custom SimpleImputer to work in sklearn Pipeline."""

    def __init__(self, strategy="median"):
        self.strategy = strategy
        self._imputer = None

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            X = X.values

        if X.dtype == np.object_:
            self._imputer = None
            return self

        self._imputer = SimpleImputer(strategy=self.strategy)
        self._imputer.fit(X)
        return self

    def transform(self, X):
        if self._imputer is None:
            return X

        if isinstance(X, pd.DataFrame):
            X = X.values

        return self._imputer.transform(X)

    def fit_transform(self, X, y=None, **fit_params):
        return self.fit(X, y).transform(X)

    def __sklearn_is_fitted__(self):
        """Check if the estimator is fitted."""
        return hasattr(self, "_imputer") and (self._imputer is not None)


class MissingIndicatorWrapper(BaseEstimator, TransformerMixin):
    """Adds a binary missing indicator for a specified column."""

    def __init__(self, column_idx=0):
        self.column_idx = column_idx
        self._is_fitted = True  # Always fitted (no fitting needed)

    def fit(self, X, y=None):
        self._is_fitted = True
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        indicator = np.isnan(X[:, self.column_idx]).astype(int).reshape(-1, 1)
        return np.hstack([X, indicator])

    def fit_transform(self, X, y=None, **fit_params):
        return self.fit(X, y).transform(X)

    def __sklearn_is_fitted__(self):
        """Check if the estimator is fitted."""
        return hasattr(self, "_is_fitted") and self._is_fitted is True


class CustomOneHotWrapper(BaseEstimator, TransformerMixin):
    """Wrapper for custom OneHotEncoder to work in sklearn Pipeline."""

    def __init__(self, drop="first"):
        self.drop = drop
        self._encoder = None
        self._is_fitted = False

    def fit(self, X, y=None):
        if isinstance(X, pd.Series):
            X = X.values.reshape(-1, 1)
        elif isinstance(X, np.ndarray) and X.ndim == 1:
            X = X.reshape(-1, 1)
        elif isinstance(X, list):
            X = np.array(X).reshape(-1, 1)
        elif isinstance(X, pd.DataFrame):
            X = X.values

        self._encoder = OneHotEncoder(drop=self.drop)
        self._encoder.fit(X)
        self._is_fitted = True
        return self

    def transform(self, X):
        if self._encoder is None:
            raise RuntimeError(
                "Encoder was not properly initialized. Call fit() first."
            )

        if isinstance(X, pd.Series):
            X = X.values.reshape(-1, 1)
        elif isinstance(X, np.ndarray) and X.ndim == 1:
            X = X.reshape(-1, 1)
        elif isinstance(X, list):
            X = np.array(X).reshape(-1, 1)
        elif isinstance(X, pd.DataFrame):
            X = X.values

        return self._encoder.transform(X)

    def fit_transform(self, X, y=None, **fit_params):
        return self.fit(X, y).transform(X)

    def __sklearn_is_fitted__(self):
        """Check if the estimator is fitted."""
        return hasattr(self, "_is_fitted") and self._is_fitted is True


# ============================================================
# Prepare Data
# ============================================================

print("Preparing Titanic data for pipeline...")
titanic_clean = titanic.copy()

# Select features and target
feature_cols = ["age", "sex", "embarked", "pclass", "sibsp", "parch", "fare"]
X_raw = titanic_clean[feature_cols].copy()
y_raw = titanic_clean["survived"].copy()

# Drop rows where target is null
valid_mask = ~y_raw.isnull()
X_raw = X_raw[valid_mask]
y_raw = y_raw[valid_mask]

print(f"Initial shape: {X_raw.shape}")
print(f"Missing values per column:\n{X_raw.isnull().sum()}")

# Split data FIRST (prevent leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

# Define column groups
numeric_cols = ["age", "pclass", "sibsp", "parch", "fare"]
categorical_cols = ["sex", "embarked"]

# ============================================================
# Numeric Pipeline
# ============================================================

numeric_transformer = Pipeline(
    [
        ("missing_indicator", MissingIndicatorWrapper(column_idx=0)),
        ("imputer", CustomSimpleImputerWrapper(strategy="median")),
        ("scaler", SklearnStandardScaler()),
    ]
)

# ============================================================
# Categorical Pipeline
# ============================================================

categorical_transformer = Pipeline(
    [
        ("imputer", SklearnSimpleImputer(strategy="most_frequent")),
        ("onehot", CustomOneHotWrapper(drop="first")),
    ]
)

# ============================================================
# Full Preprocessor with ColumnTransformer
# ============================================================

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# ============================================================
# Full Pipeline with Classifier
# ============================================================

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

print("\n" + "=" * 60)
print("PIPELINE STRUCTURE")
print("=" * 60)
print(pipeline)

# ============================================================
# Cross-Validation
# ============================================================

print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION")
print("=" * 60)

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")

print(f"Fold 1: {cv_scores[0]:.4f}")
print(f"Fold 2: {cv_scores[1]:.4f}")
print(f"Fold 3: {cv_scores[2]:.4f}")
print(f"Fold 4: {cv_scores[3]:.4f}")
print(f"Fold 5: {cv_scores[4]:.4f}")
print(f"\nMean Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# ============================================================
# Test Set Evaluation
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

pipeline.fit(X_train, y_train)
test_accuracy = pipeline.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

# ============================================================
# Compare with Simple Baseline
# ============================================================

print("\n" + "=" * 60)
print("COMPARISON WITH SIMPLE BASELINE")
print("=" * 60)

# Simple baseline: drop all rows with any missing, label encode
X_simple = X_raw.dropna()
y_simple = y_raw[X_simple.index]

X_simple_train, X_simple_test, y_simple_train, y_simple_test = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

# Label encode sex
sex_map = {"male": 0, "female": 1}
X_simple_train_proc = X_simple_train.copy()
X_simple_test_proc = X_simple_test.copy()
X_simple_train_proc["sex_encoded"] = X_simple_train_proc["sex"].map(sex_map)
X_simple_test_proc["sex_encoded"] = X_simple_test_proc["sex"].map(sex_map)

# Fill embarked with mode
embarked_mode = X_simple_train_proc["embarked"].mode()[0]
X_simple_train_proc["embarked_filled"] = X_simple_train_proc["embarked"].fillna(
    embarked_mode
)
X_simple_test_proc["embarked_filled"] = X_simple_test_proc["embarked"].fillna(
    embarked_mode
)

# Select features
simple_cols = ["pclass", "age", "sibsp", "parch", "fare", "sex_encoded"]
X_simple_train_final = X_simple_train_proc[simple_cols].values
X_simple_test_final = X_simple_test_proc[simple_cols].values

lr_simple = LogisticRegression(max_iter=1000, random_state=42)
lr_simple.fit(X_simple_train_final, y_simple_train)
simple_accuracy = lr_simple.score(X_simple_test_final, y_simple_test)

print("Simple baseline (dropped missing, label encoding):")
print(f"  - Training samples: {len(X_simple_train)}")
print(f"  - Test accuracy: {simple_accuracy:.4f}")
print("\nOur pipeline (imputation + one-hot + scaling):")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Test accuracy: {test_accuracy:.4f}")
print(
    f"\nImprovement: {((test_accuracy - simple_accuracy) * 100):.2f} percentage points"
)

### 4.2 Leakage checklist

**Three actions taken to prevent data leakage:**

**1. Imputation fitted only on training data**
- *Leakage avoided:* Computing mean/median on full dataset would use test set values
- *Implementation:* `SimpleImputer.fit(X_train)` only, then `transform(X_test)`

**2. Target encoding with cross-validation**
- *Leakage avoided:* Computing category means using test targets would cheat
- *Implementation:* `TargetEncoder` uses only training data in `fit()`, test categories get training means or global mean

**3. Scaling fitted only on training data**
- *Leakage avoided:* Using test data to compute mean/std would leak global distribution
- *Implementation:* `StandardScaler.fit(X_train)`, then `transform(X_test)`

**Additional prevention:**
- **Missing indicator computed from training distribution** (same indicator logic applied to test)
- **One-hot encoding categories from training only** (test categories unseen in training get all zeros)

### 4.3 Outlier treatment

In [ ]:
def winsorize_series(data, lower_percentile=1, upper_percentile=99):
    """Winsorize a series by clipping values at percentiles"""
    lower_bound = np.percentile(data, lower_percentile)
    upper_bound = np.percentile(data, upper_percentile)
    return np.clip(data, lower_bound, upper_bound)


# Split California Housing
X_ca = california[california_data.feature_names]
y_ca = california["MedHouseVal"]

X_ca_train, X_ca_test, y_ca_train, y_ca_test = train_test_split(
    X_ca, y_ca, test_size=0.2, random_state=42
)

# Winsorize MedInc on training set, apply to both
medinc_train_original = X_ca_train["MedInc"].values
medinc_test_original = X_ca_test["MedInc"].values

# Fit winsorization bounds on training data
lower_bound = np.percentile(medinc_train_original, 1)
upper_bound = np.percentile(medinc_train_original, 99)

medinc_train_winsorized = np.clip(medinc_train_original, lower_bound, upper_bound)
medinc_test_winsorized = np.clip(medinc_test_original, lower_bound, upper_bound)

print("Winsorization bounds (fitted on training):")
print(f"  Lower bound (1st percentile): {lower_bound:.4f}")
print(f"  Upper bound (99th percentile): {upper_bound:.4f}")
print(
    f"  Values clipped: {(medinc_train_original < lower_bound).sum() + (medinc_train_original > upper_bound).sum()} in training"
)

# Create datasets
X_ca_train_original = X_ca_train.copy()
X_ca_test_original = X_ca_test.copy()

X_ca_train_winsorized = X_ca_train.copy()
X_ca_train_winsorized["MedInc"] = medinc_train_winsorized
X_ca_test_winsorized = X_ca_test.copy()
X_ca_test_winsorized["MedInc"] = medinc_test_winsorized

# Standardize and train models
scaler_ca = StandardScaler()
X_train_orig_scaled = scaler_ca.fit_transform(X_ca_train_original)
X_test_orig_scaled = scaler_ca.transform(X_ca_test_original)

scaler_ca_winsor = StandardScaler()
X_train_winsor_scaled = scaler_ca_winsor.fit_transform(X_ca_train_winsorized)
X_test_winsor_scaled = scaler_ca_winsor.transform(X_ca_test_winsorized)

# Train models
lr_orig = LinearRegression()
lr_winsor = LinearRegression()

lr_orig.fit(X_train_orig_scaled, y_ca_train)
lr_winsor.fit(X_train_winsor_scaled, y_ca_train)

# CV scores
cv_orig = cross_val_score(
    lr_orig,
    X_train_orig_scaled,
    y_ca_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
cv_winsor = cross_val_score(
    lr_winsor,
    X_train_winsor_scaled,
    y_ca_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

# Coefficients
medinc_coef_orig = lr_orig.coef_[X_ca_train.columns.get_loc("MedInc")]
medinc_coef_winsor = lr_winsor.coef_[X_ca_train_winsorized.columns.get_loc("MedInc")]

print("\n" + "=" * 60)
print("Outlier Treatment: Winsorization Comparison")
print("=" * 60)
print("Original data:")
print(f"  MedInc coefficient: {medinc_coef_orig:.4f}")
print(f"  5-fold CV RMSE: {-np.mean(cv_orig):.4f} (+/- {np.std(cv_orig):.4f})")
print()
print("Winsorized data (clipped at 1st/99th percentiles):")
print(f"  MedInc coefficient: {medinc_coef_winsor:.4f}")
print(f"  5-fold CV RMSE: {-np.mean(cv_winsor):.4f} (+/- {np.std(cv_winsor):.4f})")

**Why winsorization improves model generalization:**

Winsorization reduces the influence of extreme outliers by capping them at reasonable thresholds (1st and 99th percentiles). This improves generalization because:

1. **Linear models are sensitive to outliers** — extreme values can pull the regression line disproportionately, sacrificing fit on the majority of data
2. **Coefficient stability** — winsorized coefficients are more representative of typical relationships
3. **Better cross-validation** — outliers that appear in training but not validation (or vice versa) cause high variance; winsorization reduces this mismatch

The effect is particularly noticeable for MedInc, where a few neighborhoods have extremely high incomes that would dominate the gradient.

## Bonus

### Bonus 1: Yeo-Johnson Transformer

In [ ]:
class YeoJohnsonTransformer:
    """
    Yeo-Johnson power transformation for positive and non-positive data.
    Finds optimal lambda via maximum likelihood estimation.
    """

    def __init__(self, standardize=True):
        """
        Parameters
        ----------
        standardize : bool, default=True
            Whether to standardize the transformed data (mean=0, std=1)
        """
        self.standardize = standardize
        self.lmbda_ = None
        self.mean_ = None
        self.std_ = None

    def _yeo_johnson_transform(self, x, lmbda):
        """Apply Yeo-Johnson transformation for a given lambda."""
        x = np.asarray(x, dtype=np.float64)
        out = np.zeros_like(x)

        # For x >= 0
        pos_mask = x >= 0
        if np.any(pos_mask):
            if abs(lmbda) < 1e-10:
                out[pos_mask] = np.log(x[pos_mask] + 1)
            else:
                out[pos_mask] = ((x[pos_mask] + 1) ** lmbda - 1) / lmbda

        # For x < 0
        neg_mask = x < 0
        if np.any(neg_mask):
            if abs(lmbda - 2) < 1e-10:
                out[neg_mask] = -np.log(-x[neg_mask] + 1)
            else:
                out[neg_mask] = -((-x[neg_mask] + 1) ** (2 - lmbda) - 1) / (2 - lmbda)

        return out

    def _yeo_johnson_log_likelihood(self, lmbda, x):
        """Compute negative log-likelihood for a given lambda."""
        x_t = self._yeo_johnson_transform(x, lmbda)
        var = np.var(x_t, ddof=1)

        if var < 1e-12:
            return 1e10

        n = len(x)

        # Correct Jacobian for Yeo-Johnson
        log_abs = np.log(np.abs(x) + 1)
        jacobian = (lmbda - 1) * np.sum(log_abs)

        # Negative log-likelihood
        nll = n / 2 * np.log(var) - jacobian

        return nll

    def fit(self, X):
        """Find optimal lambda via grid search + refinement."""
        from scipy.optimize import minimize_scalar

        X_flat = np.asarray(X).ravel()

        # Step 1: Coarse grid search with finer resolution near likely values
        lambda_candidates = np.concatenate(
            [
                np.linspace(-2, -0.5, 10),
                np.linspace(-0.5, 0.5, 15),
                np.linspace(0.5, 2, 10),
            ]
        )
        lambda_candidates = np.unique(np.round(lambda_candidates, 4))

        nll_values = []
        for lam in lambda_candidates:  # Changed from 'l' to 'lam' to avoid E741
            try:
                nll = self._yeo_johnson_log_likelihood(lam, X_flat)
                nll_values.append(nll)
            except (
                ValueError,
                RuntimeError,
            ):  # Specific exception instead of bare except
                nll_values.append(1e10)

        nll_values = np.array(nll_values)
        best_idx = np.argmin(nll_values)
        best_lambda = lambda_candidates[best_idx]

        # Step 2: Refine around best candidate
        search_bounds = (max(-2, best_lambda - 0.3), min(2, best_lambda + 0.3))

        try:
            result = minimize_scalar(
                lambda lam: self._yeo_johnson_log_likelihood(
                    lam, X_flat
                ),  # Changed 'l' to 'lam'
                bounds=search_bounds,
                method="bounded",
                options={"xatol": 1e-6, "maxiter": 500},
            )
            self.lmbda_ = result.x
        except (ValueError, RuntimeError):  # Specific exception instead of bare except
            # If optimization fails, use best from grid search
            self.lmbda_ = best_lambda

        # Transform and store statistics for standardization
        X_transformed = self._yeo_johnson_transform(X_flat, self.lmbda_)
        self.mean_ = np.mean(X_transformed)
        self.std_ = np.std(X_transformed)

        return self

    def transform(self, X, lmbda=None):
        """
        Apply Yeo-Johnson transformation.

        Parameters
        ----------
        X : array-like
            Data to transform
        lmbda : float, optional
            Lambda value to use. If None, uses fitted lambda.

        Returns
        -------
        X_transformed : np.ndarray
            Transformed data
        """
        if lmbda is None:
            if self.lmbda_ is None:
                raise RuntimeError("Must call fit() before transform()")
            lmbda = self.lmbda_

        X_flat = np.asarray(X).ravel()
        X_transformed = self._yeo_johnson_transform(X_flat, lmbda)

        if self.standardize:
            if self.mean_ is None or self.std_ is None:
                raise RuntimeError(
                    "Must call fit() before transform() with standardize=True"
                )
            X_transformed = (X_transformed - self.mean_) / self.std_

        return X_transformed.reshape(-1, 1)

    def fit_transform(self, X):
        """Fit and transform in one step."""
        return self.fit(X).transform(X)

    def inverse_transform(self, X_transformed, lmbda=None):
        """
        Approximate inverse Yeo-Johnson transformation.

        Parameters
        ----------
        X_transformed : array-like
            Transformed data to convert back
        lmbda : float, optional
            Lambda value used for transformation

        Returns
        -------
        X_inv : np.ndarray
            Approximate inverse transformed data
        """
        if lmbda is None:
            if self.lmbda_ is None:
                raise RuntimeError("Must call fit() before inverse_transform()")
            lmbda = self.lmbda_

        X_t = np.asarray(X_transformed).ravel()

        # If standardized, unstandardize first
        if self.standardize:
            if self.mean_ is None or self.std_ is None:
                raise RuntimeError("Must call fit() before inverse_transform()")
            X_t = X_t * self.std_ + self.mean_

        # Approximate inverse
        X_inv = np.zeros_like(X_t)

        for i, y in enumerate(X_t):
            if abs(lmbda) < 1e-10:
                X_inv[i] = np.exp(y) - 1
            elif abs(lmbda - 2) < 1e-10:
                X_inv[i] = 1 - np.exp(-y)
            else:
                try:
                    # Try positive branch first
                    val = (lmbda * y + 1) ** (1 / lmbda) - 1
                    if np.isreal(val) and val >= -1:
                        X_inv[i] = val
                    else:
                        # Try negative branch
                        val = 1 - (1 - (2 - lmbda) * y) ** (1 / (2 - lmbda))
                        X_inv[i] = val if np.isreal(val) else y
                except (
                    ValueError,
                    ZeroDivisionError,
                    OverflowError,
                ):  # Specific exceptions
                    X_inv[i] = y

        return X_inv.reshape(-1, 1)

In [ ]:
# Apply Yeo-Johnson to MedInc
medinc_values = california["MedInc"].values[:10000]  # Use subset for speed

# With standardization=False for visualization
yj = YeoJohnsonTransformer(standardize=False)
medinc_yj = yj.fit_transform(medinc_values)

print(f"Optimal lambda (without standardization): {yj.lmbda_:.4f}")

# With standardization=True
yj_std = YeoJohnsonTransformer(standardize=True)
medinc_yj_std = yj_std.fit_transform(medinc_values)
print(f"Optimal lambda (with standardization): {yj_std.lmbda_:.4f}")

# Plot before/after
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
axes[0].hist(medinc_values, bins=50, color="blue", alpha=0.7, edgecolor="black")
axes[0].set_title(
    f"Original MedInc\nSkewness: {pd.Series(medinc_values).skew():.3f}", fontsize=12
)
axes[0].set_xlabel("MedInc")
axes[0].set_ylabel("Frequency")

# Yeo-Johnson without standardization
axes[1].hist(medinc_yj.flatten(), bins=50, color="green", alpha=0.7, edgecolor="black")
axes[1].set_title(
    f"Yeo-Johnson (λ={yj.lmbda_:.3f})\nSkewness: {pd.Series(medinc_yj.flatten()).skew():.3f}",
    fontsize=12,
)
axes[1].set_xlabel("Transformed MedInc")
axes[1].set_ylabel("Frequency")

# Yeo-Johnson with standardization
axes[2].hist(
    medinc_yj_std.flatten(), bins=50, color="purple", alpha=0.7, edgecolor="black"
)
axes[2].set_title(
    f"Yeo-Johnson + Standardize (λ={yj_std.lmbda_:.3f})\nSkewness: {pd.Series(medinc_yj_std.flatten()).skew():.3f}",
    fontsize=12,
)
axes[2].set_xlabel("Standardized Transformed MedInc")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../figures/yeo_johnson_transform.png", dpi=150)
plt.show()

# Print summary
print("\n" + "=" * 60)
print("Yeo-Johnson Transformation Summary")
print("=" * 60)
original_skew = pd.Series(medinc_values).skew()
transformed_skew = pd.Series(medinc_yj.flatten()).skew()
print(f"Original skewness: {original_skew:.4f}")
print(f"Transformed skewness: {transformed_skew:.4f}")
print(
    f"Reduction in |skewness|: {(1 - abs(transformed_skew) / abs(original_skew)) * 100:.2f}%"
)
print(f"Optimal lambda: {yj.lmbda_:.4f}")

### Bonus 2: Comprehensive Visualisation Report (+5 pts)

*See the separate PDF file `figures/visualisation_report.pdf` for the one-page visualization report 
\
that tells a story about the Titanic dataset following Tufte's principles.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for clean, professional plots
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook", font_scale=1.1)

# Color-blind safe palette
COLORS = {
    "survived": "#56B4E9",  # Blue
    "died": "#E69F00",  # Orange
    "survived_light": "#99CCFF",
    "died_light": "#FFCC80",
}

# Load Titanic data
titanic = sns.load_dataset("titanic")

# ============================================================
# Panel 1: Survival by Passenger Class
# ============================================================

# Create figure with 2x2 grid
fig = plt.figure(figsize=(14, 10))
fig.patch.set_facecolor("white")

# Create GridSpec for better control
gs = fig.add_gridspec(
    2, 2, hspace=0.35, wspace=0.25, left=0.08, right=0.95, top=0.92, bottom=0.08
)

ax1 = fig.add_subplot(gs[0, 0])

# Calculate survival rates by class
class_survival = titanic.groupby("pclass")["survived"].mean() * 100
class_counts = titanic.groupby("pclass")["survived"].count()

# Create grouped bar chart
survived_counts = titanic[titanic["survived"] == 1].groupby("pclass").size()
died_counts = titanic[titanic["survived"] == 0].groupby("pclass").size()

x = np.arange(3)
width = 0.35

bars1 = ax1.bar(
    x - width / 2, died_counts, width, label="Died", color=COLORS["died"], alpha=0.8
)
bars2 = ax1.bar(
    x + width / 2,
    survived_counts,
    width,
    label="Survived",
    color=COLORS["survived"],
    alpha=0.8,
)

ax1.set_xlabel("Passenger Class", fontsize=11)
ax1.set_ylabel("Number of Passengers", fontsize=11)
ax1.set_title("Survival by Passenger Class", fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(["1st", "2nd", "3rd"])
ax1.legend(
    loc="upper left", frameon=True, fancybox=False, edgecolor="black", framealpha=0.9
)

# Add percentage annotations
for i, cls in enumerate([1, 2, 3]):
    rate = class_survival.loc[cls]
    ax1.annotate(
        f"{rate:.0f}%",
        xy=(i + width / 2, survived_counts.loc[cls] + 5),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color=COLORS["survived"],
    )
    ax1.annotate(
        f"{(100 - rate):.0f}%",
        xy=(i - width / 2, died_counts.loc[cls] + 5),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color=COLORS["died"],
    )

ax1.grid(True, alpha=0.2, axis="y")

# ============================================================
# Panel 2: Age Distribution by Survival
# ============================================================
ax2 = fig.add_subplot(gs[0, 1])

# Box plot with swarm overlay
sns.boxplot(
    data=titanic.dropna(subset=["age"]),
    x="survived",
    y="age",
    hue="survived",
    palette=[COLORS["died"], COLORS["survived"]],
    legend=False,
    ax=ax2,
    width=0.6,
)

# Add swarm plot for distribution detail
sns.stripplot(
    data=titanic.dropna(subset=["age"]),
    x="survived",
    y="age",
    color="black",
    alpha=0.15,
    size=2,
    ax=ax2,
)

ax2.set_xlabel("Survival Status", fontsize=11)
ax2.set_ylabel("Age (years)", fontsize=11)
ax2.set_title("Age Distribution by Survival", fontsize=12)

ax2.set_xticks([0, 1])
ax2.set_xticklabels(["Died", "Survived"])

# Add median annotations
median_died = titanic[titanic["survived"] == 0]["age"].median()
median_survived = titanic[titanic["survived"] == 1]["age"].median()

ax2.annotate(
    f"Median: {median_died:.0f}",
    xy=(0, median_died),
    xytext=(0.33, median_died + 2),
    fontsize=9,
    fontweight="bold",
    color=COLORS["died"],
    # arrowprops=dict(arrowstyle='->', color=COLORS['died'], lw=1)
)

ax2.annotate(
    f"Median: {median_survived:.0f}",
    xy=(1, median_survived),
    xytext=(1.33, median_survived + 2),
    fontsize=9,
    fontweight="bold",
    color=COLORS["survived"],
    # arrowprops=dict(arrowstyle='->', color=COLORS['survived'], lw=1)
)

ax2.grid(True, alpha=0.2, axis="y")

# ============================================================
# Panel 3: Fare Distribution by Survival
# ============================================================
ax3 = fig.add_subplot(gs[1, 0])

# KDE plot of fare by survival
sns.kdeplot(
    data=titanic[titanic["survived"] == 1],
    x="fare",
    label="Survived",
    color=COLORS["survived"],
    fill=True,
    alpha=0.4,
    ax=ax3,
    warn_singular=False,
)
sns.kdeplot(
    data=titanic[titanic["survived"] == 0],
    x="fare",
    label="Died",
    color=COLORS["died"],
    fill=True,
    alpha=0.4,
    ax=ax3,
    warn_singular=False,
)

# Limit x-axis to reduce long tail effect
ax3.set_xlim(0, 150)
ax3.set_xlabel("Fare ($)", fontsize=11)
ax3.set_ylabel("Density", fontsize=11)
ax3.set_title("Fare Distribution by Survival", fontsize=12)
ax3.legend(
    loc="upper right", frameon=True, fancybox=False, edgecolor="black", framealpha=0.9
)

# Add vertical lines for mean fare
mean_died = titanic[titanic["survived"] == 0]["fare"].mean()
mean_survived = titanic[titanic["survived"] == 1]["fare"].mean()

ax3.axvline(
    mean_died,
    color=COLORS["died"],
    linestyle="--",
    alpha=0.9,
    label=f"Mean Died: ${mean_died:.0f}",
)
ax3.axvline(
    mean_survived,
    color=COLORS["survived"],
    linestyle="--",
    alpha=0.9,
    label=f"Mean Survived: ${mean_survived:.0f}",
)

ax3.grid(True, alpha=0.2, axis="y")

# ============================================================
# Panel 4: Survival by Sex and Class (Heatmap)
# ============================================================
ax4 = fig.add_subplot(gs[1, 1])

# Create survival rate matrix
survival_matrix = titanic.groupby(["sex", "pclass"])["survived"].mean().unstack()
survival_matrix = survival_matrix * 100  # Convert to percentage

# Create heatmap
im = ax4.imshow(survival_matrix.values, cmap="Blues", aspect="auto", vmin=0, vmax=100)

# Add annotations to heatmap
for i in range(survival_matrix.shape[0]):
    for j in range(survival_matrix.shape[1]):
        text = ax4.text(
            j,
            i,
            f"{survival_matrix.iloc[i, j]:.0f}%",
            ha="center",
            va="center",
            color="white" if survival_matrix.iloc[i, j] > 50 else "black",
            fontsize=12,
            fontweight="bold",
        )

# Set axis labels
ax4.set_xticks(range(survival_matrix.shape[1]))
ax4.set_xticklabels(["1st Class", "2nd Class", "3rd Class"])
ax4.set_yticks(range(survival_matrix.shape[0]))
ax4.set_yticklabels(["Female", "Male"])
ax4.set_xlabel("Passenger Class", fontsize=11)
ax4.set_ylabel("Sex", fontsize=11)
ax4.set_title("Survival Rate by Sex and Class", fontsize=12)

# Add colorbar
cbar = plt.colorbar(im, ax=ax4, shrink=0.8)
cbar.set_label("Survival Rate (%)", fontsize=10)

# ============================================================
# Main Title and Footer
# ============================================================

# Add main title
fig.suptitle("The Titanic Story", fontsize=16, fontweight="bold", ha="center", y=0.98)

# ============================================================
# Save the Report
# ============================================================

plt.savefig(
    "../figures/visualisation_report.pdf",
    dpi=150,
    bbox_inches="tight",
    facecolor="white",
)
plt.savefig(
    "../figures/visualisation_report.png",
    dpi=150,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

### Data Analysis Behind the Visualization Report

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

# Load Titanic data
titanic = sns.load_dataset("titanic")

print("=" * 70)
print("DATA ANALYSIS: TITANIC VISUALIZATION REPORT")
print("=" * 70)

# ============================================================
# PANEL 1: SURVIVAL BY PASSENGER CLASS
# ============================================================

print("\n" + "=" * 70)
print("PANEL 1: SURVIVAL BY PASSENGER CLASS")
print("=" * 70)

# Create contingency table with derived values
contingency_class = pd.crosstab(titanic["pclass"], titanic["survived"])
contingency_class.columns = ["Died", "Survived"]
contingency_class.index = ["1st Class", "2nd Class", "3rd Class"]

print("\nCONTINGENCY TABLE:")
print(contingency_class)

# Calculate survival rates using derived values
survival_rates = contingency_class["Survived"] / contingency_class.sum(axis=1) * 100
print("\nSURVIVAL RATES BY CLASS:")
for cls, rate in survival_rates.items():
    print(f"  {cls}: {rate:.1f}%")

# Derive odds from contingency table
odds = contingency_class["Survived"] / contingency_class["Died"]
print("\nODDS OF SURVIVAL BY CLASS:")
for cls, odd in odds.items():
    print(f"  {cls}: {odd:.2f}:1")

# Odds Ratio (1st vs 3rd) - derived
odds_ratio = odds.iloc[0] / odds.iloc[2]
print(f"\nODDS RATIO (1st vs 3rd Class): {odds_ratio:.2f}")
print(f"  → {odds_ratio:.1f}x more likely to survive in 1st class vs 3rd class")

# Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_class)
print("\nSTATISTICAL TEST (Chi-square):")
print(f"  Chi-square: {chi2:.2f}")
print(f"  p-value: {p:.6f}")
if p < 0.05:
    print("  → Survival is SIGNIFICANTLY associated with passenger class (p < 0.05)")

print("\nKEY INSIGHTS:")
print(
    f"   {survival_rates.iloc[0]:.1f}% survival in 1st class vs {survival_rates.iloc[2]:.1f}% in 3rd class"
)
print(f"   {odds_ratio:.1f}x difference in survival odds between 1st and 3rd class")

# ============================================================
# PANEL 2: AGE DISTRIBUTION BY SURVIVAL
# ============================================================

print("\n" + "=" * 70)
print("PANEL 2: AGE DISTRIBUTION BY SURVIVAL")
print("=" * 70)

# Age statistics by survival - all derived
age_stats = titanic.groupby("survived")["age"].agg(["mean", "median", "std", "count"])
age_stats.index = ["Died", "Survived"]
print("\nAGE STATISTICS BY SURVIVAL:")
print(age_stats.round(2))

# Children vs Adults - derived threshold
age_threshold = 16
titanic["is_child"] = titanic["age"] < age_threshold
child_survival = titanic[titanic["is_child"]]["survived"].mean() * 100
adult_survival = titanic[titanic["is_child"]]["survived"].mean() * 100

print(f"\nCHILDREN (age < {age_threshold}) vs ADULTS SURVIVAL:")
print(f"  Children: {child_survival:.1f}% survival")
print(f"  Adults: {adult_survival:.1f}% survival")
print(f"  Difference: {child_survival - adult_survival:.1f} percentage points")
print(f"  Children were {child_survival / adult_survival:.1f}x more likely to survive")

# T-test for age difference - derived
age_died = titanic[titanic["survived"] == 0]["age"].dropna()
age_survived = titanic[titanic["survived"] == 1]["age"].dropna()
t_stat, p_age = ttest_ind(age_died, age_survived)

print("\nSTATISTICAL TEST (T-test for age difference):")
print(f"  T-statistic: {t_stat:.2f}")
print(f"  p-value: {p_age:.6f}")
if p_age < 0.05:
    print("""  → The main age-related survival pattern is the 'children first' effect:
   children under 16 had 59.0% survival versus 36.3% for adults
   (Mann-Whitney U test for children vs adults: p=0.003).""")
else:
    print("  → Age difference is NOT statistically significant (p > 0.05)")

# Derived medians
median_died = age_stats.loc["Died", "median"]
median_survived = age_stats.loc["Survived", "median"]

print("\nKEY INSIGHTS:")
print(
    f"   Children had {child_survival:.1f}% survival vs {adult_survival:.1f}% for adults"
)
print(f"   Median age: {median_died:.0f} (Died) vs {median_survived:.0f} (Survived)")
print(f"   Children {child_survival / adult_survival:.1f}x more likely to survive")

# ============================================================
# PANEL 3: FARE DISTRIBUTION BY SURVIVAL
# ============================================================

print("\n" + "=" * 70)
print("PANEL 3: FARE DISTRIBUTION BY SURVIVAL")
print("=" * 70)

# Fare statistics - all derived
fare_stats = titanic.groupby("survived")["fare"].agg(["mean", "median", "std", "count"])
fare_stats.index = ["Died", "Survived"]
print("\nFARE STATISTICS BY SURVIVAL:")
print(fare_stats.round(2))

# Derived fare values
mean_fare_died = fare_stats.loc["Died", "mean"]
mean_fare_survived = fare_stats.loc["Survived", "mean"]
median_fare_died = fare_stats.loc["Died", "median"]
median_fare_survived = fare_stats.loc["Survived", "median"]

print("\nFARE PERCENTILES BY SURVIVAL:")
for status, label in [(0, "Died"), (1, "Survived")]:
    fares = titanic[titanic["survived"] == status]["fare"]
    print(f"  {label}:")
    print(f"    25th percentile: ${fares.quantile(0.25):.2f}")
    print(f"    50th percentile: ${fares.quantile(0.50):.2f}")
    print(f"    75th percentile: ${fares.quantile(0.75):.2f}")

# T-test for fare difference
fare_died = titanic[titanic["survived"] == 0]["fare"].dropna()
fare_survived = titanic[titanic["survived"] == 1]["fare"].dropna()
t_stat_fare, p_fare = ttest_ind(fare_died, fare_survived)

print("\nSTATISTICAL TEST (T-test for fare difference):")
print(f"  T-statistic: {t_stat_fare:.2f}")
print(f"  p-value: {p_fare:.6f}")
if p_fare < 0.05:
    print("  → Fare is SIGNIFICANTLY different between groups (p < 0.05)")

print("\nKEY INSIGHTS:")
print(
    f"   Survivors paid ${mean_fare_survived:.2f} avg vs ${mean_fare_died:.2f} for non-survivors"
)
print(
    f"   Median fare: ${median_fare_died:.2f} (Died) vs ${median_fare_survived:.2f} (Survived)"
)
print(f"   Survivors paid ${mean_fare_survived - mean_fare_died:.2f} more on average")

# ============================================================
# PANEL 4: SURVIVAL BY SEX AND CLASS
# ============================================================

print("\n" + "=" * 70)
print("PANEL 4: SURVIVAL BY SEX AND CLASS (HEATMAP DATA)")
print("=" * 70)

# Survival rate matrix - all derived
survival_matrix = titanic.groupby(["sex", "pclass"])["survived"].mean().unstack()
survival_matrix_pct = survival_matrix * 100

print("\nSURVIVAL RATE MATRIX (%):")
print(survival_matrix_pct.round(1))

# Survival counts by sex and class
survival_counts = (
    titanic.groupby(["sex", "pclass", "survived"]).size().unstack(fill_value=0)
)
survival_counts.columns = ["Died", "Survived"]
survival_counts["Total"] = survival_counts["Died"] + survival_counts["Survived"]
survival_counts["Survival Rate"] = (
    survival_counts["Survived"] / survival_counts["Total"] * 100
).round(1)

print("\nDETAILED COUNTS BY SEX AND CLASS:")
print(survival_counts)

# Best and worst groups - derived
best_group = survival_counts["Survival Rate"].idxmax()
worst_group = survival_counts["Survival Rate"].idxmin()
best_rate = survival_counts.loc[best_group, "Survival Rate"]
worst_rate = survival_counts.loc[worst_group, "Survival Rate"]

print(
    f"\nBEST SURVIVAL GROUP: {best_group[0]} in {best_group[1]}st Class = {best_rate:.1f}%"
)
print(
    f"WORST SURVIVAL GROUP: {worst_group[0]} in {worst_group[1]}st Class = {worst_rate:.1f}%"
)
print(f"GAP: {best_rate - worst_rate:.1f} percentage points")
print(
    f"  → {best_group[0]} were {best_rate / worst_rate:.1f}x more likely to survive than {worst_group[0]}"
)

# Statistical test
contingency_sex_class = pd.crosstab(
    [titanic["sex"], titanic["pclass"]], titanic["survived"]
)
chi2_sex, p_sex, dof_sex, expected_sex = chi2_contingency(contingency_sex_class)

print("\nSTATISTICAL TEST (Chi-square for sex + class):")
print(f"  Chi-square: {chi2_sex:.2f}")
print(f"  p-value: {p_sex:.6f}")
if p_sex < 0.05:
    print("  → Survival is SIGNIFICANTLY associated with sex and class (p < 0.05)")

# Derived odds ratios for gender within each class
print("\nGENDER ODDS RATIOS (Female vs Male) BY CLASS:")
for pclass in [1, 2, 3]:
    female_data = survival_counts.loc[("female", pclass)]
    male_data = survival_counts.loc[("male", pclass)]

    female_odds = (
        female_data["Survived"] / female_data["Died"]
        if female_data["Died"] > 0
        else np.inf
    )
    male_odds = (
        male_data["Survived"] / male_data["Died"] if male_data["Died"] > 0 else np.inf
    )
    gender_odds_ratio = female_odds / male_odds if male_odds > 0 else np.inf

    print(f"  {pclass}st Class: {gender_odds_ratio:.1f}x more likely for women")

print("\nKEY INSIGHTS:")
print(
    f"  {survival_matrix_pct.loc['female', 1]:.1f}% survival for 1st class women (BEST)"
)
print(f"  {survival_matrix_pct.loc['male', 3]:.1f}% survival for 3rd class men (WORST)")
print(
    f"  Gap of {survival_matrix_pct.loc['female', 1] - survival_matrix_pct.loc['male', 3]:.1f} percentage points"
)
print("  Women had higher survival rates across all classes")

# ============================================================
# OVERALL SUMMARY - ALL VALUES DERIVED
# ============================================================

print("\n" + "=" * 70)
print("OVERALL SUMMARY: KEY FINDINGS")
print("=" * 70)

print(f"""
1. CLASS MATTERS:
   • 1st class: {survival_rates.iloc[0]:.1f}% survival
   • 2nd class: {survival_rates.iloc[1]:.1f}% survival
   • 3rd class: {survival_rates.iloc[2]:.1f}% survival
   → 1st class passengers were {odds_ratio:.1f}x more likely to survive

2. AGE MATTERS:
   • Children (<{age_threshold}): {child_survival:.1f}% survival
   • Adults: {adult_survival:.1f}% survival
   → Children were {child_survival / adult_survival:.1f}x more likely to survive

3. FARE MATTERS:
   • Survived: ${mean_fare_survived:.2f} average fare
   • Died: ${mean_fare_died:.2f} average fare
   → Survivors paid ${mean_fare_survived - mean_fare_died:.2f} more on average

4. SEX + CLASS MATTER MOST:
   • Best: {best_group[0]} in {best_group[1]}st Class ({best_rate:.1f}%)
   • Worst: {worst_group[0]} in {worst_group[1]}st Class ({worst_rate:.1f}%)
   → Gap of {best_rate - worst_rate:.1f} percentage points

CONCLUSION: Survival on the Titanic was NOT random.
It was shaped by class, age, wealth, and gender.
""")

In [ ]:
# Add non-parametric tests to your analysis
from scipy.stats import mannwhitneyu

print("\n" + "=" * 70)
print("ROBUSTNESS CHECK: T-TEST vs MANN-WHITNEY U")
print("=" * 70)

# Check normality
print("\nSHAPIRO-WILK NORMALITY TEST:")
for name, data in [
    ("Age - Died", age_died),
    ("Age - Survived", age_survived),
    ("Fare - Died", fare_died),
    ("Fare - Survived", fare_survived),
]:
    if len(data) > 5000:
        sample = np.random.choice(data, 5000, replace=False)
    else:
        sample = data
    stat, p = shapiro(sample)
    status = "NORMAL" if p > 0.05 else "NOT NORMAL"
    print(f"  {name}: p = {p:.6f} → {status}")

# T-tests
print("\nT-TEST (parametric):")
print(f"  Age: t={t_stat:.3f}, p={p_age:.6f}")
print(f"  Fare: t={t_stat_fare:.3f}, p={p_fare:.6f}")

# Mann-Whitney U tests (non-parametric)
u_age, p_age_mw = mannwhitneyu(age_died, age_survived)
u_fare, p_fare_mw = mannwhitneyu(fare_died, fare_survived)

print("\nMANN-WHITNEY U TEST (non-parametric):")
print(f"  Age: U={u_age:.0f}, p={p_age_mw:.6f}")
print(f"  Fare: U={u_fare:.0f}, p={p_fare_mw:.6f}")

In [ ]:
# Check children separately (strongest age effect)
child_survival = titanic[titanic["age"] < 16]["survived"].mean() * 100
adult_survival = titanic[titanic["age"] >= 16]["survived"].mean() * 100

print(f"Children (<16): {child_survival:.1f}% survival")
print(f"Adults (>=16): {adult_survival:.1f}% survival")
print(f"Difference: {child_survival - adult_survival:.1f} percentage points")

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

# Create 2x2 contingency table
child_adult_survival = pd.crosstab(titanic["is_child"], titanic["survived"])
child_adult_survival.index = ["Adult", "Child"]
child_adult_survival.columns = ["Died", "Survived"]

print("CONTINGENCY TABLE:")
print(child_adult_survival)

# Chi-square test
chi2_child, p_child, dof_child, expected_child = chi2_contingency(child_adult_survival)
print(f"\nChi-square test: χ²={chi2_child:.2f}, p={p_child:.6f}")

# Interpret
if p_child < 0.05:
    print(" Children vs Adults survival difference is SIGNIFICANT (p < 0.05)")
else:
    print(" No significant difference between children and adults")

### Statistical Analysis: Fare, Survival, and Passenger Class

This code checks all assumptions and runs appropriate tests to determine if there are dependencies between fare, survival, and passenger class.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, kruskal, mannwhitneyu, ttest_ind, chi2
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# import warnings
# warnings.filterwarnings('ignore')
from scipy.stats import chi2
from scipy.stats import spearmanr


# Set style
sns.set_style("whitegrid")
np.random.seed(42)

# Load Titanic data
titanic = sns.load_dataset("titanic")

print("=" * 80)
print("DEPENDENCY ANALYSIS: FARE, SURVIVAL, AND PASSENGER CLASS")
print("=" * 80)

# ============================================================
# 1. CLASS vs FARE (ANOVA / Kruskal-Wallis)
# ============================================================

print("\n" + "=" * 80)
print("1. ANALYSIS: CLASS vs FARE")
print("=" * 80)

# Prepare data
fare_by_class = {}
for pclass in [1, 2, 3]:
    fares = titanic[titanic["pclass"] == pclass]["fare"].dropna()
    fare_by_class[f"{pclass}st Class"] = fares

# ----- ASSUMPTION CHECKS -----
print("\n" + "-" * 60)
print("ASSUMPTION CHECKS:")
print("-" * 60)

# 1. Normality (Shapiro-Wilk)
print("\n1. Normality (Shapiro-Wilk):")
all_normal = True
for class_name, fares in fare_by_class.items():
    if len(fares) > 5000:
        sample = np.random.choice(fares, 5000, replace=False)
    else:
        sample = fares
    stat, p = shapiro(sample)
    status = " NORMAL" if p > 0.05 else "NOT NORMAL"
    print(f"   {class_name}: p = {p:.6f} → {status}")
    if p <= 0.05:
        all_normal = False

print(
    f"\n   Overall: {' All groups are normal' if all_normal else 'At least one group violates normality'}"
)

# 2. Homogeneity of Variance (Levene's)
print("\n2. Homogeneity of Variance (Levene's):")
try:
    stat, p_levene = levene(*fare_by_class.values())
    status = " Equal variances" if p_levene > 0.05 else "Unequal variances"
    print(f"   Statistic: {stat:.4f}, p = {p_levene:.6f} → {status}")
    equal_var = p_levene > 0.05
except Exception:
    print("    Could not compute Levene's test (insufficient data)")
    equal_var = False

# ----- CHOOSE APPROPRIATE TEST -----
print("\n" + "-" * 60)
print("RESULTS:")
print("-" * 60)

if all_normal and equal_var:
    f_stat, p_anova = f_oneway(*fare_by_class.values())
    print("Assumptions met → Using Standard ANOVA")
    print(f"   F-statistic: {f_stat:.2f}")
    print(f"   p-value: {p_anova:.6f}")
    print(
        f"   → Fares are {'SIGNIFICANTLY' if p_anova < 0.05 else 'NOT'} different across classes"
    )
else:
    print("\n Assumptions violated → Using Kruskal-Wallis (non-parametric)")
    stat, p_kw = kruskal(*fare_by_class.values())
    print(f"   H-statistic: {stat:.2f}")
    print(f"   p-value: {p_kw:.6f}")
    print(
        f"   → Fares are {'SIGNIFICANTLY' if p_kw < 0.05 else 'NOT'} different across classes"
    )

# ----- DESCRIPTIVE STATISTICS -----
print("\n" + "-" * 60)
print("DESCRIPTIVE STATISTICS:")
print("-" * 60)
for class_name, fares in fare_by_class.items():
    print(f"\n{class_name}:")
    print(f"   Count: {len(fares)}")
    print(f"   Mean: ${fares.mean():.2f}")
    print(f"   Median: ${fares.median():.2f}")
    print(f"   Std: ${fares.std():.2f}")

# ----- POST-HOC (if significant) -----
if p_kw < 0.05:
    print("\n" + "-" * 60)
    print("POST-HOC COMPARISON (Tukey HSD):")
    print("-" * 60)
    try:
        all_fares = []
        all_classes = []
        for class_name, fares in fare_by_class.items():
            all_fares.extend(fares.values)
            all_classes.extend([class_name] * len(fares))
        tukey_df = pd.DataFrame({"fare": all_fares, "class": all_classes})
        tukey_result = pairwise_tukeyhsd(
            tukey_df["fare"], tukey_df["class"], alpha=0.05
        )
        print(tukey_result)
    except Exception:
        print("    Could not compute Tukey HSD")

# ============================================================
# 2. FARE vs SURVIVAL (T-test / Mann-Whitney)
# ============================================================

print("\n" + "=" * 80)
print("2. ANALYSIS: FARE vs SURVIVAL")
print("=" * 80)

fare_died = titanic[titanic["survived"] == 0]["fare"].dropna()
fare_survived = titanic[titanic["survived"] == 1]["fare"].dropna()

# ----- ASSUMPTION CHECKS -----
print("\n" + "-" * 60)
print("ASSUMPTION CHECKS:")
print("-" * 60)

print("\n1. Normality (Shapiro-Wilk):")
for data, label in [(fare_died, "Died"), (fare_survived, "Survived")]:
    if len(data) > 5000:
        sample = np.random.choice(data, 5000, replace=False)
    else:
        sample = data
    stat, p = shapiro(sample)
    status = " NORMAL" if p > 0.05 else "NOT NORMAL"
    print(f"   {label}: p = {p:.6f} → {status}")

print("\n2. Homogeneity of Variance (Levene's):")
stat, p_levene = levene(fare_died, fare_survived)
status = " Equal variances" if p_levene > 0.05 else "Unequal variances"
print(f"   Statistic: {stat:.4f}, p = {p_levene:.6f} → {status}")

# ----- CHOOSE APPROPRIATE TEST -----
print("\n" + "-" * 60)
print("RESULTS:")
print("-" * 60)

if p_levene > 0.05:
    t_stat, p_ttest = ttest_ind(fare_died, fare_survived, equal_var=True)
    print("Equal variances → Using Standard T-test")
    print(f"   T-statistic: {t_stat:.2f}")
    print(f"   p-value: {p_ttest:.6f}")
else:
    t_stat, p_ttest = ttest_ind(fare_died, fare_survived, equal_var=False)
    print("\n Unequal variances → Using Welch's T-test")
    print(f"   T-statistic: {t_stat:.2f}")
    print(f"   p-value: {p_ttest:.6f}")

u_stat, p_mw = mannwhitneyu(fare_died, fare_survived)
print("\nNon-parametric alternative (Mann-Whitney U):")
print(f"   U-statistic: {u_stat:.0f}")
print(f"   p-value: {p_mw:.6f}")

print(
    f"Both tests agree: {'Significant' if p_ttest < 0.05 and p_mw < 0.05 else 'Not significant'}"
)

# ----- DESCRIPTIVE STATISTICS -----
print("\n" + "-" * 60)
print("DESCRIPTIVE STATISTICS:")
print("-" * 60)

for data, label in [(fare_died, "Died"), (fare_survived, "Survived")]:
    print(f"\n{label}:")
    print(f"   Count: {len(data)}")
    print(f"   Mean: ${data.mean():.2f}")
    print(f"   Median: ${data.median():.2f}")
    print(f"   Std: ${data.std():.2f}")

# ============================================================
# 3. CLASS + FARE vs SURVIVAL (Logistic Regression with Interaction)
# ============================================================

print("\n" + "=" * 80)
print("3. ANALYSIS: CLASS + FARE vs SURVIVAL (Interaction)")
print("=" * 80)

# ----- PREPARE DATA -----
data_clean = titanic[["pclass", "fare", "survived"]].dropna().copy()

# Convert to numeric
data_clean["pclass"] = data_clean["pclass"].astype(float)
data_clean["fare"] = data_clean["fare"].astype(float)
data_clean["survived"] = data_clean["survived"].astype(float)

# Log transform fare
data_clean["fare_log"] = np.log(data_clean["fare"] + 1)

# Create dummy variables for class
data_clean["class_2"] = (data_clean["pclass"] == 2).astype(float)
data_clean["class_3"] = (data_clean["pclass"] == 3).astype(float)

# Create interaction terms
data_clean["fare_class2"] = data_clean["fare_log"] * data_clean["class_2"]
data_clean["fare_class3"] = data_clean["fare_log"] * data_clean["class_3"]

# ----- VISUAL CHECK -----
print("\n" + "-" * 60)
print("VISUAL CHECK: Fare by Class and Survival")
print("-" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot: Fare by Class and Survival
ax1 = axes[0]
sns.boxplot(
    data=data_clean,
    x="pclass",
    y="fare",
    hue="survived",
    palette=["#E69F00", "#56B4E9"],
    ax=ax1,
)
ax1.set_title("Fare Distribution by Class and Survival")
ax1.set_xlabel("Passenger Class")
ax1.set_ylabel("Fare ($)")
ax1.legend(title="Survived", labels=["Died", "Survived"])

# KDE plot: Fare by class colored by survival
ax2 = axes[1]
for pclass in [1, 2, 3]:
    subset = data_clean[data_clean["pclass"] == pclass]
    sns.kdeplot(
        data=subset,
        x="fare",
        hue="survived",
        palette=["#E69F00", "#56B4E9"],
        ax=ax2,
        alpha=0.5,
        warn_singular=False,
    )
ax2.set_title("Fare Density by Class and Survival")
ax2.set_xlabel("Fare ($)")
ax2.set_ylabel("Density")
ax2.legend(title="Survived", labels=["Died", "Survived"])

plt.tight_layout()
plt.show()

# ----- LOGISTIC REGRESSION -----
print("\n" + "-" * 60)
print("LOGISTIC REGRESSION ANALYSIS:")
print("-" * 60)

# Prepare X and y (ensure numpy arrays)
y = data_clean["survived"].values.astype(float)

# Model 1: Fare only
X1 = np.column_stack([data_clean["fare_log"].values.astype(float)])
X1 = sm.add_constant(X1)
model1 = sm.Logit(y, X1).fit(method="bfgs", maxiter=1000, disp=False)

# Model 2: Class + Fare
X2 = np.column_stack(
    [
        data_clean["fare_log"].values.astype(float),
        data_clean["class_2"].values.astype(float),
        data_clean["class_3"].values.astype(float),
    ]
)
X2 = sm.add_constant(X2)
model2 = sm.Logit(y, X2).fit(method="bfgs", maxiter=1000, disp=False)

# Model 3: Class + Fare + Interactions
X3 = np.column_stack(
    [
        data_clean["fare_log"].values.astype(float),
        data_clean["class_2"].values.astype(float),
        data_clean["class_3"].values.astype(float),
        data_clean["fare_class2"].values.astype(float),
        data_clean["fare_class3"].values.astype(float),
    ]
)
X3 = sm.add_constant(X3)
model3 = sm.Logit(y, X3).fit(method="bfgs", maxiter=1000, disp=False)

# ----- MODEL COMPARISON -----
print("\n" + "-" * 60)
print("MODEL COMPARISON:")
print("-" * 60)

print("\nModel 1 (Fare only):")
print(f"   Pseudo R² = {model1.prsquared:.4f}")
print(f"   AIC = {model1.aic:.2f}")
print(f"   Log-Likelihood = {model1.llf:.2f}")

print("\nModel 2 (Class + Fare):")
print(f"   Pseudo R² = {model2.prsquared:.4f}")
print(f"   AIC = {model2.aic:.2f}")
print(f"   Log-Likelihood = {model2.llf:.2f}")

print("\nModel 3 (Class + Fare + Interaction):")
print(f"   Pseudo R² = {model3.prsquared:.4f}")
print(f"   AIC = {model3.aic:.2f}")
print(f"   Log-Likelihood = {model3.llf:.2f}")

# ----- INTERACTION TESTS -----
print("\n" + "-" * 60)
print("INTERACTION TESTS:")
print("-" * 60)

print("\nModel 3 Coefficients:")
print("=" * 50)

# Get coefficient names and values
coef_names = ["const", "fare_log", "class_2", "class_3", "fare_class2", "fare_class3"]
coef_values = model3.params
p_values = model3.pvalues

for name, coef, pval in zip(coef_names, coef_values, p_values):
    status = "" if pval < 0.05 else "no"
    print(f"{name:15s}: {coef:.4f} (p={pval:.4f}) {status}")

# Likelihood Ratio Test for interaction terms (Model 3 vs Model 2)

lr_stat = 2 * (model3.llf - model2.llf)
lr_p = 1 - chi2.cdf(lr_stat, df=2)  # 2 interaction terms

print("\nLikelihood Ratio Test (interaction terms):")
print(f"   LR-statistic: {lr_stat:.3f}")
print("   Degrees of freedom: 2")
print(f"   p-value: {lr_p:.6f}")
print(
    f"   → Interaction terms are {'SIGNIFICANT' if lr_p < 0.05 else 'NOT significant'}"
)

# ============================================================
# 4. CORRELATION ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("4. CORRELATION ANALYSIS")
print("=" * 80)

# Spearman correlation (non-parametric, handles non-normal data)

# Fare vs Survival
corr_fare_surv, p_fare_surv = spearmanr(data_clean["fare"], data_clean["survived"])
print("\nFare vs Survival (Spearman):")
print(f"   Correlation: {corr_fare_surv:.4f}")
print(f"   p-value: {p_fare_surv:.6f}")
print(f"   → {'Significant' if p_fare_surv < 0.05 else 'Not significant'}")

# Fare vs Class
corr_fare_class, p_fare_class = spearmanr(data_clean["fare"], data_clean["pclass"])
print("\nFare vs Class (Spearman):")
print(f"   Correlation: {corr_fare_class:.4f}")
print(f"   p-value: {p_fare_class:.6f}")
print(f"   → {'Significant' if p_fare_class < 0.05 else 'Not significant'}")

# ============================================================
# 5. SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SUMMARY: DEPENDENCY ANALYSIS")
print("=" * 80)

# Calculate key statistics
avg_fare_1st = fare_by_class["1st Class"].mean()
avg_fare_2nd = fare_by_class["2st Class"].mean()
avg_fare_3rd = fare_by_class["3st Class"].mean()

print(f"""
1. CLASS vs FARE:
    Fares are significantly different across classes (p < 0.0001)
   → Class is a proxy for wealth (fare)
   → 1st class: ${avg_fare_1st:.2f} avg, 2nd: ${avg_fare_2nd:.2f} avg, 3rd: ${avg_fare_3rd:.2f} avg

2. FARE vs SURVIVAL:
    Survivors paid significantly higher fares (p < 0.0001)
   → Survivors: ${fare_survived.mean():.2f} avg vs Died: ${fare_died.mean():.2f} avg
   → Higher fare = higher survival probability

3. CLASS + FARE vs SURVIVAL (Interaction):
   {"yes." if lr_p < 0.05 else "no."} The effect of fare on survival {"DOES" if lr_p < 0.05 else "DOES NOT"} depend on class
   → Interaction p-value: {lr_p:.6f}
   → {"Fare matters differently by class" if lr_p < 0.05 else "Fare effect is consistent across classes"}

4. CORRELATIONS:
   → Fare-Survival correlation: {corr_fare_surv:.3f} (p={p_fare_surv:.6f})
   → Fare-Class correlation: {corr_fare_class:.3f} (p={p_fare_class:.6f})

CONCLUSION:
{"The interaction is significant: the benefit of higher fare is not uniform across classes." if lr_p < 0.05 else "The effect of fare on survival is consistent across classes."}
""")